# CLIP-ViT + LoRA Continual Learning on Split CIFAR-100

This notebook implements a compact, reproducible comparison setup.

Main setup:

- CLIP-ViT vision encoder: `openai/clip-vit-base-patch16`
- Split CIFAR-100 continual learning
- 5 steps, 20 classes per step
.

Main comparison:

1. `seq_ft_no_replay`
2. `simple_avg_no_replay`
3. `simple_avg_replay`
4. `do_merging_simple`
5. `orthogonal_loss`
6. `rank_extension`
7. `joint_upper_bound`


In [ ]:
# ============================================================
# 1) Imports
# ============================================================

import os
import gc
import json
import random
import math
import inspect
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, concatenate_datasets
from torchvision import transforms

from transformers import (
    CLIPImageProcessor,
    CLIPVisionModel,
    TrainingArguments,
    Trainer,
    set_seed,
)

from transformers.modeling_outputs import ImageClassifierOutput

from peft import LoraConfig, get_peft_model

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

In [ ]:
# ============================================================
# 2) Full comparison configuration
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEBUG_MODE = False

FAST_RUN = False

RUN_NAME_BASE = "clip_vit_lora_cifar100_full_comparison_with_orth_rankext"
RUN_NAME = f"{RUN_NAME_BASE}_{'FAST_RUN_DEBUG' if FAST_RUN else 'EPOCH3_MAIN'}"

MODEL_CHECKPOINT = "openai/clip-vit-base-patch16"

NUM_CLASSES = 100
NUM_STEPS = 5
CLASSES_PER_STEP = 20

# ============================================================
# Epoch setup
# ============================================================
FULL_FT_EPOCHS = 3
FULL_LORA_EPOCHS = 3
FULL_JOINT_EPOCHS = 3
FULL_ORTH_EPOCHS = 3
FULL_RANKEXT_EPOCHS = 3

SCRATCH_EPOCHS = 1

if FAST_RUN:
    FT_EPOCHS = SCRATCH_EPOCHS
    LORA_EPOCHS = SCRATCH_EPOCHS
    JOINT_EPOCHS = SCRATCH_EPOCHS
    ORTH_EPOCHS = SCRATCH_EPOCHS
    RANKEXT_EPOCHS = SCRATCH_EPOCHS
else:
    FT_EPOCHS = FULL_FT_EPOCHS
    LORA_EPOCHS = FULL_LORA_EPOCHS
    JOINT_EPOCHS = FULL_JOINT_EPOCHS
    ORTH_EPOCHS = FULL_ORTH_EPOCHS
    RANKEXT_EPOCHS = FULL_RANKEXT_EPOCHS

# ============================================================
# Batch settings
# ============================================================

BATCH_FT = 8
ACCUM_FT = 2

BATCH_LORA = 16
ACCUM_LORA = 1

# ============================================================
# Learning rates
# ============================================================

LR_FT = 3e-5
LR_LORA = 5e-5
LR_JOINT = 5e-5

LR_ORTH = 5e-5

LR_RANKEXT = 1e-4

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# ============================================================
# LoRA setup for CLIP-ViT
# ============================================================

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# ============================================================
# Small replay diagnostic beside simple average
# ============================================================

REPLAY_PER_CLASS = 20
RANKEXT_REPLAY_PER_CLASS = REPLAY_PER_CLASS

# ============================================================
# Orthogonal-loss setup
# ============================================================
# Experimental guidance (paper-aligned context, no setup change):
# - pretrained CLIP-ViT backbone
# - Split CIFAR-100 class-incremental evaluation
# - keep all_seen as primary continual-learning indicator
# Loss = CE + lambda_orth * mean_i tr(M_(t-1)^i (Delta W_t^i)^T)

LAMBDA_ORTH = 0.03
ORTH_LOSS_TYPE = "trace"  # one of: trace, qr_lorac, combined
ORTH_SCALE_MODE = "squared_trace"  # one of: raw_trace, abs_trace, squared_trace, cosine, auto_ratio
ORTH_TARGET_RATIO = 0.01
ORTH_LAMBDA_MIN = 1e-6
ORTH_LAMBDA_MAX = 1e3
ORTH_EPS = 1e-12
USE_IPC_CONSTRAINT = False
LAMBDA_IPC = 0.0
IPC_TOP_P = 0.10
IPC_IMPORTANCE_NUM_BATCHES = 8

# LoRAC/IPC-inspired QR orthogonality and important-parameter constraint
# Lightweight approximation, not exact full reproduction.
ORTH_CONFIG_SWEEP = [
    {"orth_loss_type": "trace", "lambda_orth": 0.0, "use_ipc": False, "lambda_ipc": 0.0, "orth_scale_mode": "squared_trace", "orth_target_ratio": 0.0},
    {"orth_loss_type": "trace", "lambda_orth": 1.0, "use_ipc": False, "lambda_ipc": 0.0, "orth_scale_mode": "auto_ratio", "orth_target_ratio": 0.001},
    {"orth_loss_type": "trace", "lambda_orth": 1.0, "use_ipc": False, "lambda_ipc": 0.0, "orth_scale_mode": "auto_ratio", "orth_target_ratio": 0.01},
    {"orth_loss_type": "trace", "lambda_orth": 1.0, "use_ipc": False, "lambda_ipc": 0.0, "orth_scale_mode": "auto_ratio", "orth_target_ratio": 0.05},
]
ENABLE_RANKEXT_ORTH_CONFIG_SWEEP = True
ORTH_DIAGNOSTICS = True
RANKEXT_DIAGNOSTICS = True

# ============================================================
# Rank-extension setup
# ============================================================
# Step 1: rank 8
# Step 2: rank 16, old rank 8 copied and frozen
# Step 3: rank 24, old rank 16 copied and frozen
# Instead use alpha_t = RANKEXT_ALPHA_PER_RANK * total_rank
# so alpha_t / total_rank stays constant.

RANKEXT_NEW_RANK_PER_STEP = 8
RANKEXT_ALPHA_PER_RANK = 2.0

# ============================================================
# Next run focus: DO-Merging + orthogonality optimization
# Keep all methods implemented for reproducibility, but disable unrelated runs.
# ============================================================

METHODS_TO_RUN = {
    "seq_ft_no_replay": False,
    "simple_avg_no_replay": False,
    "simple_avg_replay": False,
    "do_merging_simple": False,
    "orthogonal_loss": False,
    "rank_extension": False,
    "rank_extension_replay": False,
    "rank_extension_orth": True,
    "rank_extension_replay_orth": False,
    "joint_upper_bound": False,
}

# ============================================================
# Fixed output path on cluster
# ============================================================

ROOT_RESULTS_DIR = "/nfsd/lttm4/tesisti/shahrampour/projects/runs/cifar100_5step_n5_main/results"

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_OUTPUT_DIR = os.path.join(
    ROOT_RESULTS_DIR,
    f"{RUN_NAME}_{RUN_TAG}"
)

TABLES_DIR = os.path.join(BASE_OUTPUT_DIR, "tables")
PLOTS_DIR = os.path.join(BASE_OUTPUT_DIR, "plots")
MODELS_DIR = os.path.join(BASE_OUTPUT_DIR, "models")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

all_results = []

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("FP16:", USE_FP16)
print("Checkpoint:", MODEL_CHECKPOINT)
print("BASE_OUTPUT_DIR:", BASE_OUTPUT_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("PLOTS_DIR:", PLOTS_DIR)
print("MODELS_DIR:", MODELS_DIR)

print("\nRun mode:")
print({
    "FAST_RUN": FAST_RUN,
    "RUN_NAME": RUN_NAME,
    "SCRATCH_EPOCHS": SCRATCH_EPOCHS,
})

print("\nEpochs:")
print({
    "FT_EPOCHS": FT_EPOCHS,
    "LORA_EPOCHS": LORA_EPOCHS,
    "JOINT_EPOCHS": JOINT_EPOCHS,
    "ORTH_EPOCHS": ORTH_EPOCHS,
    "RANKEXT_EPOCHS": RANKEXT_EPOCHS,
    "FULL_FT_EPOCHS": FULL_FT_EPOCHS,
    "FULL_LORA_EPOCHS": FULL_LORA_EPOCHS,
    "FULL_JOINT_EPOCHS": FULL_JOINT_EPOCHS,
    "FULL_ORTH_EPOCHS": FULL_ORTH_EPOCHS,
    "FULL_RANKEXT_EPOCHS": FULL_RANKEXT_EPOCHS,
})

print("\nLoRA:")
print({
    "LORA_R": LORA_R,
    "LORA_ALPHA": LORA_ALPHA,
    "LORA_DROPOUT": LORA_DROPOUT,
    "TARGET_MODULES": TARGET_MODULES,
})

print("\nOrthogonal loss:")
print({
    "LAMBDA_ORTH": LAMBDA_ORTH,
    "ORTH_LOSS_TYPE": ORTH_LOSS_TYPE,
    "ORTH_SCALE_MODE": ORTH_SCALE_MODE,
    "ORTH_TARGET_RATIO": ORTH_TARGET_RATIO,
    "ORTH_LAMBDA_MIN": ORTH_LAMBDA_MIN,
    "ORTH_LAMBDA_MAX": ORTH_LAMBDA_MAX,
    "ORTH_EPS": ORTH_EPS,
    "USE_IPC_CONSTRAINT": USE_IPC_CONSTRAINT,
    "LAMBDA_IPC": LAMBDA_IPC,
    "IPC_TOP_P": IPC_TOP_P,
    "IPC_IMPORTANCE_NUM_BATCHES": IPC_IMPORTANCE_NUM_BATCHES,
    "ORTH_CONFIG_SWEEP": ORTH_CONFIG_SWEEP,
    "ENABLE_RANKEXT_ORTH_CONFIG_SWEEP": ENABLE_RANKEXT_ORTH_CONFIG_SWEEP,
    "ORTH_DIAGNOSTICS": ORTH_DIAGNOSTICS,
})

print("\nRank extension:")
print({
    "RANKEXT_NEW_RANK_PER_STEP": RANKEXT_NEW_RANK_PER_STEP,
    "RANKEXT_ALPHA_PER_RANK": RANKEXT_ALPHA_PER_RANK,
    "LR_RANKEXT": LR_RANKEXT,
    "RANKEXT_REPLAY_PER_CLASS": RANKEXT_REPLAY_PER_CLASS,
    "RANKEXT_DIAGNOSTICS": RANKEXT_DIAGNOSTICS,
})

print("\nMethods:")
print(json.dumps(METHODS_TO_RUN, indent=2))

enabled_methods = [m for m, enabled in METHODS_TO_RUN.items() if enabled]
print("Enabled methods for this run:", enabled_methods)

expected_enabled = {"rank_extension_orth"}
unexpected_enabled = sorted(set(enabled_methods) - expected_enabled)
missing_expected = sorted(expected_enabled - set(enabled_methods))

if unexpected_enabled:
    print("[WARNING] Unexpected enabled methods:", unexpected_enabled)
if missing_expected:
    print("[WARNING] Expected methods not enabled:", missing_expected)


In [ ]:
# ============================================================
# 3) Load CIFAR-100 and define continual splits
# ============================================================

dataset = load_dataset("cifar100")

LABEL_COL = "fine_label" if "fine_label" in dataset["train"].column_names else "label"
IMAGE_COL = "img" if "img" in dataset["train"].column_names else "image"

class_splits = [
    list(range(0, 20)),
    list(range(20, 40)),
    list(range(40, 60)),
    list(range(60, 80)),
    list(range(80, 100)),
]

first_step_classes = class_splits[0]
later_step_classes = [c for split in class_splits[1:] for c in split]
all_classes = [c for split in class_splits for c in split]

def classes_for_step(step_idx):
    return class_splits[step_idx]

def filter_by_classes(ds, class_ids):
    class_ids = set(class_ids)
    return ds.filter(lambda x: int(x[LABEL_COL]) in class_ids)

print("Dataset columns:", dataset["train"].column_names)
print("Label column:", LABEL_COL)
print("Image column:", IMAGE_COL)
for i, cls in enumerate(class_splits, start=1):
    print(f"Step {i}: {cls[0]}-{cls[-1]}")

In [ ]:
# ============================================================
# 4) CLIP image processor and transforms
# ============================================================

image_processor = CLIPImageProcessor.from_pretrained(MODEL_CHECKPOINT)

if hasattr(image_processor, "crop_size") and image_processor.crop_size is not None:
    H = int(image_processor.crop_size.get("height", 224))
    W = int(image_processor.crop_size.get("width", 224))
else:
    H = W = 224

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.05,
        contrast=0.05,
        saturation=0.05,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = np.squeeze(x).astype(np.uint8)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {
        "pixel_values": pixel_values,
        "labels": labels,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": float((preds == labels).mean())
    }

print("Image size:", H, W)
print("CLIP mean:", image_processor.image_mean)
print("CLIP std:", image_processor.image_std)

In [ ]:
# ============================================================
# 5) CLIP-ViT vision classifier wrapper
# ============================================================

class CLIPVisionForCIFAR100(nn.Module):
    """
    CLIP-ViT vision encoder + trainable CIFAR-100 classifier.

    This uses:
    openai/clip-vit-base-patch16

    The text encoder is not used.
    Only the CLIP vision backbone is used.
    """

    def __init__(self, checkpoint, num_labels):
        super().__init__()

        self.vision_model = CLIPVisionModel.from_pretrained(checkpoint)

        hidden_size = self.vision_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

        self.config = self.vision_model.config
        self.config.num_labels = num_labels
        self.config.id2label = {i: str(i) for i in range(num_labels)}
        self.config.label2id = {str(i): i for i in range(num_labels)}

    def forward(
        self,
        pixel_values=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=True,
        **kwargs,
    ):
        outputs = self.vision_model(
            pixel_values=pixel_values,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return ImageClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

def fresh_pretrained_model():
    """
    Fresh CLIP-ViT vision model with a CIFAR-100 classifier.
    """
    return CLIPVisionForCIFAR100(
        checkpoint=MODEL_CHECKPOINT,
        num_labels=NUM_CLASSES,
    )

def disable_incompatible_torchao_for_peft():
    """
    Colab may have an old torchao installed. Recent PEFT checks torchao during
    LoRA injection and raises before falling back to normal nn.Linear LoRA.
    This guard disables only PEFT's torchao LoRA dispatcher when that version
    check fails; it does not change the LoRA method.
    """
    try:
        import peft.import_utils as peft_import_utils

        try:
            peft_import_utils.is_torchao_available()
            return
        except ImportError as e:
            if "incompatible version of torchao" not in str(e):
                raise

        peft_import_utils.is_torchao_available = lambda: False

        try:
            import peft.tuners.lora.torchao as peft_lora_torchao
            peft_lora_torchao.is_torchao_available = lambda: False
        except Exception:
            pass

        print(
            "[PEFT compatibility] Disabled torchao LoRA dispatcher "
            "because installed torchao is incompatible with PEFT."
        )
    except ImportError:
        return

def add_lora(model):
    """
    Add LoRA to CLIP-ViT attention q_proj and v_proj modules.
    """
    disable_incompatible_torchao_for_peft()

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        modules_to_save=["classifier"],
    )

    model = get_peft_model(model, lora_config)
    return model

print("LoRA target modules:", TARGET_MODULES)


In [ ]:
# ============================================================
# 6) Dataset helpers
# ============================================================

def build_replay_dataset(old_classes, replay_per_class):
    if len(old_classes) == 0 or replay_per_class <= 0:
        return None

    parts = []

    for cls in old_classes:
        cls_ds = filter_by_classes(dataset["train"], [cls])
        n = min(replay_per_class, len(cls_ds))
        cls_ds = cls_ds.shuffle(seed=SEED).select(range(n))
        parts.append(cls_ds)

    replay_ds = concatenate_datasets(parts)
    return replay_ds

def make_train_dataset(step_idx, replay_per_class=0):
    current_classes = classes_for_step(step_idx)
    current_ds = filter_by_classes(dataset["train"], current_classes)

    old_classes = []
    for old_step in range(step_idx):
        old_classes.extend(classes_for_step(old_step))

    replay_ds = build_replay_dataset(
        old_classes=old_classes,
        replay_per_class=replay_per_class,
    )

    if replay_ds is None:
        final_ds = current_ds
    else:
        final_ds = concatenate_datasets([current_ds, replay_ds])

    final_ds = final_ds.shuffle(seed=SEED + step_idx)
    final_ds = final_ds.with_transform(preprocess_train)

    print(
        f"Step {step_idx + 1} | "
        f"current={len(current_ds)} | "
        f"replay={0 if replay_ds is None else len(replay_ds)} | "
        f"total={len(final_ds)}"
    )

    return final_ds

def make_eval_dataset(class_ids):
    ds = filter_by_classes(dataset["test"], class_ids)
    ds = ds.with_transform(preprocess_val)
    return ds

def make_joint_train_dataset():
    ds = dataset["train"].shuffle(seed=SEED)
    ds = ds.with_transform(preprocess_train)
    return ds

def make_joint_eval_dataset():
    ds = dataset["test"]
    ds = ds.with_transform(preprocess_val)
    return ds

eval_first = make_eval_dataset(first_step_classes)
eval_later = make_eval_dataset(later_step_classes)
eval_all_seen = make_eval_dataset(all_classes)

print("first_step eval:", len(eval_first))
print("later_steps eval:", len(eval_later))
print("all_seen eval:", len(eval_all_seen))

In [ ]:
# ============================================================
# 7) Trainer helpers
# ============================================================

def get_training_args(
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    train_dataset_len=None,
    eval_strategy="epoch",
):
    """
    Trainer settings.

    We use warmup_steps instead of warmup_ratio because warmup_ratio is deprecated
    in newer Transformers versions.
    """

    if train_dataset_len is not None:
        steps_per_epoch = math.ceil(train_dataset_len / batch_size / accum_steps)
        total_steps = int(steps_per_epoch * epochs)
        warmup_steps = int(WARMUP_RATIO * total_steps)
    else:
        warmup_steps = 0

    kwargs = dict(
        output_dir=output_dir,
        remove_unused_columns=False,
        save_strategy="no",
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=accum_steps,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        report_to="none",
        max_grad_norm=1.0,
    )

    sig = inspect.signature(TrainingArguments.__init__)

    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = eval_strategy
    else:
        kwargs["evaluation_strategy"] = eval_strategy

    return TrainingArguments(**kwargs)

def train_with_trainer(
    model,
    train_ds,
    eval_ds,
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    trainer_cls=Trainer,
    **trainer_kwargs,
):
    args = get_training_args(
        output_dir=output_dir,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        accum_steps=accum_steps,
        train_dataset_len=len(train_ds),
        eval_strategy="epoch",
    )

    trainer = trainer_cls(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        **trainer_kwargs,
    )

    trainer.train()
    eval_out = trainer.evaluate()

    return trainer, eval_out

def evaluate_model(model, method_name):
    args = get_training_args(
        output_dir=os.path.join(MODELS_DIR, f"eval_{method_name}"),
        epochs=1,
        lr=LR_LORA,
        batch_size=BATCH_LORA,
        accum_steps=ACCUM_LORA,
        train_dataset_len=None,
        eval_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    eval_first_out = trainer.evaluate(eval_dataset=eval_first)
    eval_later_out = trainer.evaluate(eval_dataset=eval_later)
    eval_all_out = trainer.evaluate(eval_dataset=eval_all_seen)

    rows = [
        {
            "method": method_name,
            "eval_set": "first_step",
            "accuracy": float(eval_first_out["eval_accuracy"]),
            "loss": float(eval_first_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "later_steps",
            "accuracy": float(eval_later_out["eval_accuracy"]),
            "loss": float(eval_later_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "all_seen",
            "accuracy": float(eval_all_out["eval_accuracy"]),
            "loss": float(eval_all_out["eval_loss"]),
        },
    ]

    all_results.extend(rows)

    print(pd.DataFrame(rows))
    return rows

In [ ]:
# ============================================================
# 8) LoRA extraction and merge helpers
# ============================================================

def normalize_module_name(name):
    prefixes = [
        "base_model.model.",
        "model.",
    ]

    out = name

    for p in prefixes:
        if out.startswith(p):
            out = out[len(p):]

    return out

def extract_lora_state(model):
    """
    Extract:
    - LoRA delta_W per target module
    - classifier weights

    PEFT convention:
    A shape = [r, in_features]
    B shape = [out_features, r]
    delta_W = B @ A * scaling
    """
    state = {
        "deltas": {},
        "lora_A": {},
        "lora_B": {},
        "scaling": {},
        "classifier_weight": None,
        "classifier_bias": None,
    }

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        adapter_name = "default"
        A = module.lora_A[adapter_name].weight.detach().cpu().float().clone()
        B = module.lora_B[adapter_name].weight.detach().cpu().float().clone()

        scaling = (
            module.scaling[adapter_name]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        scaling = float(scaling)
        delta = scaling * (B @ A)

        plain_name = normalize_module_name(name)
        state["deltas"][plain_name] = delta.clone()
        state["lora_A"][plain_name] = A
        state["lora_B"][plain_name] = B
        state["scaling"][plain_name] = scaling

    for name, tensor in model.state_dict().items():
        if "classifier.modules_to_save.default.weight" in name:
            state["classifier_weight"] = tensor.detach().cpu().clone()

        if "classifier.modules_to_save.default.bias" in name:
            state["classifier_bias"] = tensor.detach().cpu().clone()

    return state

def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)

    current = model

    for part in module_name.split("."):
        if part == "":
            continue
        current = getattr(current, part)

    return current

def simple_average_deltas(step_states):
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    for key in keys:
        vals = []

        for state in step_states:
            if key in state["deltas"]:
                vals.append(state["deltas"][key].float())

        merged[key] = torch.stack(vals, dim=0).mean(dim=0)

    return merged

def column_normalize(mat, eps=1e-12):
    return mat / torch.linalg.norm(mat, dim=0, keepdim=True).clamp_min(eps)

def column_decouple_delta(delta, eps=1e-12):
    magnitudes = torch.linalg.norm(delta, dim=0, keepdim=True).clamp_min(eps)
    directions = delta / magnitudes
    return magnitudes, directions

def mean_pairwise_cosine(flat_vectors, eps=1e-12):
    if len(flat_vectors) < 2:
        return None

    sims = []

    for i in range(len(flat_vectors)):
        vi = flat_vectors[i]
        vi = vi / torch.linalg.norm(vi).clamp_min(eps)

        for j in range(i + 1, len(flat_vectors)):
            vj = flat_vectors[j]
            vj = vj / torch.linalg.norm(vj).clamp_min(eps)
            sims.append(torch.dot(vi, vj).item())

    if len(sims) == 0:
        return None

    return float(sum(sims) / len(sims))

def orthogonalize_task_directions(task_deltas, eps=1e-12):
    mags = []
    dirs = []
    flat_dirs = []

    for delta in task_deltas:
        mag, direction = column_decouple_delta(delta, eps=eps)
        mags.append(mag)
        dirs.append(direction)
        flat_dirs.append(direction.reshape(-1))

    ortho_flat = []

    for v in flat_dirs:
        u = v.clone()

        for q in ortho_flat:
            u = u - torch.dot(u, q) * q

        n = torch.linalg.norm(u)

        if n < eps:
            u = v / torch.linalg.norm(v).clamp_min(eps)
        else:
            u = u / n

        ortho_flat.append(u)

    ortho_dirs = [
        column_normalize(u.reshape_as(dirs[i]), eps=eps)
        for i, u in enumerate(ortho_flat)
    ]

    return mags, ortho_dirs

def do_merge_deltas(step_states, eps=1e-12, use_orthogonalize=True, verbose=True):
    """
    DO-Merging-inspired: layer-wise orthogonalized, column-wise decoupled LoRA delta merging.
    """
    all_keys = sorted(set().union(*[set(s["deltas"].keys()) for s in step_states]))
    merged = {}

    layer_delta_counts = []
    cos_before_values = []
    cos_after_values = []
    col_mag_means = []
    col_mag_stds = []

    for key in all_keys:
        task_deltas = []

        for state in step_states:
            if key in state["deltas"]:
                task_deltas.append(state["deltas"][key].detach().cpu().float())

        if len(task_deltas) == 0:
            continue

        layer_delta_counts.append(len(task_deltas))

        mags_before = []
        dirs_before = []

        for delta in task_deltas:
            mag, direction = column_decouple_delta(delta, eps=eps)
            mags_before.append(mag)
            dirs_before.append(direction)

        flat_before = [d.reshape(-1) for d in dirs_before]
        cos_before = mean_pairwise_cosine(flat_before, eps=eps)

        if cos_before is not None:
            cos_before_values.append(cos_before)

        if len(task_deltas) == 1:
            merged[key] = task_deltas[0].clone()
            continue

        if use_orthogonalize:
            mags, dirs = orthogonalize_task_directions(task_deltas, eps=eps)
        else:
            mags = mags_before
            dirs = dirs_before

        flat_after = [d.reshape(-1) for d in dirs]
        cos_after = mean_pairwise_cosine(flat_after, eps=eps)

        if cos_after is not None:
            cos_after_values.append(cos_after)

        mag_stack = torch.stack(mags, dim=0)
        col_mag_means.append(float(mag_stack.mean().item()))
        col_mag_stds.append(float(mag_stack.std(unbiased=False).item()))

        merged_mag = mag_stack.mean(dim=0)
        merged_dir = torch.stack(dirs, dim=0).mean(dim=0)
        merged_dir = column_normalize(merged_dir, eps=eps)

        merged_delta = merged_dir * merged_mag

        if merged_delta.shape != task_deltas[0].shape:
            raise ValueError(
                f"Shape mismatch for {key}: merged={tuple(merged_delta.shape)} vs ref={tuple(task_deltas[0].shape)}"
            )

        merged[key] = merged_delta

    if verbose:
        print(f"[DO-Merging] merged {len(merged)} layers")

        if len(layer_delta_counts) > 0:
            mean_tasks = sum(layer_delta_counts) / len(layer_delta_counts)
            print(f"[DO-Merging] avg task deltas per layer: {mean_tasks:.2f}")

        if len(cos_before_values) > 0:
            print(
                f"[DO-Merging] avg pairwise cosine before orthogonalization: {sum(cos_before_values) / len(cos_before_values):.6f}"
            )
        else:
            print("[DO-Merging] avg pairwise cosine before orthogonalization: n/a")

        if len(cos_after_values) > 0:
            print(
                f"[DO-Merging] avg pairwise cosine after orthogonalization: {sum(cos_after_values) / len(cos_after_values):.6f}"
            )
        else:
            print("[DO-Merging] avg pairwise cosine after orthogonalization: n/a")

        if len(col_mag_means) > 0:
            print(
                f"[DO-Merging] column magnitude mean/std across layers: {sum(col_mag_means) / len(col_mag_means):.6f} / {sum(col_mag_stds) / len(col_mag_stds):.6f}"
            )
        else:
            print("[DO-Merging] column magnitude mean/std across layers: n/a")

    return merged

def apply_deltas_to_base(merged_deltas, step_states):
    """
    Apply merged LoRA deltas to a fresh CLIP-ViT model and stitch classifier rows.
    """
    model = fresh_pretrained_model()

    with torch.no_grad():
        for key, delta in merged_deltas.items():
            try:
                module = get_submodule_by_name(model, key)
            except Exception as e:
                print("Could not find module:", key, "|", e)
                continue

            if not hasattr(module, "weight"):
                print("Module has no weight:", key)
                continue

            module.weight.add_(
                delta.to(
                    device=module.weight.device,
                    dtype=module.weight.dtype,
                )
            )

        for step_idx, state in enumerate(step_states):
            classes = classes_for_step(step_idx)

            if state["classifier_weight"] is None:
                print("Missing classifier for step", step_idx + 1)
                continue

            w = state["classifier_weight"].to(model.classifier.weight.device)
            b = state["classifier_bias"].to(model.classifier.bias.device)

            for c in classes:
                model.classifier.weight[c].copy_(w[c])
                model.classifier.bias[c].copy_(b[c])

    return model

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 9) Baseline: seq_ft_no_replay
# ============================================================

if METHODS_TO_RUN["seq_ft_no_replay"]:
    seq_ft_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"seq_ft_no_replay_step_{step_idx + 1}",
        )

        print(
            f"\n===== seq_ft_no_replay | "
            f"step {step_idx + 1}/{NUM_STEPS} ====="
        )

        train_with_trainer(
            model=seq_ft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=FT_EPOCHS,
            lr=LR_FT,
            batch_size=BATCH_FT,
            accum_steps=ACCUM_FT,
        )

    evaluate_model(seq_ft_model, "seq_ft_no_replay")

    del seq_ft_model
    cleanup()

else:
    print("Skipping seq_ft_no_replay")

In [ ]:
# ============================================================
# 10) Train independent LoRAs
# ============================================================

def train_independent_loras(method_prefix, replay_per_class=0):
    """
    Train one independent LoRA per step.

    Used for:
    - simple_avg_no_replay
    - simple_avg_replay
    - do_merging_simple
    """
    step_states = []

    for step_idx in range(NUM_STEPS):
        model = fresh_pretrained_model()
        model = add_lora(model)

        model.print_trainable_parameters()

        train_ds = make_train_dataset(
            step_idx=step_idx,
            replay_per_class=replay_per_class,
        )

        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"{method_prefix}_step_{step_idx + 1}",
        )

        print(
            f"\n===== {method_prefix} | "
            f"step {step_idx + 1}/{NUM_STEPS} | "
            f"replay_per_class={replay_per_class} ====="
        )

        train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=LORA_EPOCHS,
            lr=LR_LORA,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
        )

        state = extract_lora_state(model)
        step_states.append(state)

        del model
        cleanup()

    return step_states

In [ ]:
# ============================================================
# 11) simple_avg_no_replay
# ============================================================

step_states_no_replay = None

if METHODS_TO_RUN["simple_avg_no_replay"] or METHODS_TO_RUN["do_merging_simple"]:
    step_states_no_replay = train_independent_loras(
        method_prefix="simple_avg_no_replay_source",
        replay_per_class=0,
    )

    if METHODS_TO_RUN["simple_avg_no_replay"]:
        simple_delta = simple_average_deltas(step_states_no_replay)
        simple_model = apply_deltas_to_base(
            merged_deltas=simple_delta,
            step_states=step_states_no_replay,
        )

        evaluate_model(simple_model, "simple_avg_no_replay")

        del simple_model
        cleanup()
    else:
        print("Prepared step_states_no_replay for do_merging_simple; skipping simple_avg_no_replay evaluation")

else:
    print("Skipping simple_avg_no_replay")

In [ ]:
# ============================================================
# 12) simple_avg_replay
# ============================================================

step_states_replay = None

if METHODS_TO_RUN["simple_avg_replay"]:
    step_states_replay = train_independent_loras(
        method_prefix="simple_avg_replay_source",
        replay_per_class=REPLAY_PER_CLASS,
    )

    replay_delta = simple_average_deltas(step_states_replay)
    replay_model = apply_deltas_to_base(
        merged_deltas=replay_delta,
        step_states=step_states_replay,
    )

    evaluate_model(replay_model, "simple_avg_replay")

    del replay_model
    cleanup()

else:
    print("Skipping simple_avg_replay")

In [ ]:
# ============================================================
# 13) do_merging_simple
# ============================================================

if METHODS_TO_RUN["do_merging_simple"]:
    assert step_states_no_replay is not None, "step_states_no_replay is required for do_merging_simple"

    do_delta = do_merge_deltas(step_states_no_replay)
    do_layer_count = len(do_delta)
    expected_do_layers = 24  # CLIP ViT-B/16: 12 blocks x (q_proj, v_proj)
    print(f"[DO-Merging] merged layer count: {do_layer_count}")
    if abs(do_layer_count - expected_do_layers) > 2:
        print(
            f"[WARNING] do_merging_simple merged {do_layer_count} layers; expected around {expected_do_layers}."
        )
    do_model = apply_deltas_to_base(
        merged_deltas=do_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(do_model, "do_merging_simple")

    del do_model
    cleanup()

else:
    print("Skipping do_merging_simple")

In [ ]:
# ============================================================
# 14) orthogonal_loss
# ============================================================

# L = CE + lambda_orth * mean_i tr(M_(t-1)^i (Delta W_t^i)^T)
# Here M_(t-1)^i is the previous absorbed model weight for the same q_proj/v_proj layer.
# Delta W_t^i is the current LoRA update B @ A for that layer.

def extract_reference_weights_for_orth(peft_model):
    """
    Extract M_(t-1) for every current LoRA target module.
    These are the base q_proj/v_proj weights before training the current LoRA.
    """
    reference_weights = {}

    for name, module in peft_model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if hasattr(module, "base_layer") and hasattr(module.base_layer, "weight"):
            reference_weights[plain_name] = module.base_layer.weight.detach().cpu().float().clone()
        elif hasattr(module, "weight"):
            reference_weights[plain_name] = module.weight.detach().cpu().float().clone()

    return reference_weights

def compute_orth_penalty(model, reference_weights, eps=1e-8):
    """
    Mean normalized trace between previous absorbed model weights and current LoRA deltas.

    Orthogonality objective per layer:
        tr(M_(t-1) DeltaW_t^T)

    Since both matrices have the same shape, this is equivalent to:
        sum(M_(t-1) * DeltaW_t)

    We normalize by matrix norms to keep the scale stable during debug runs.
    """
    penalties = []
    device = next(model.parameters()).device

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if plain_name not in reference_weights:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        delta = (B @ A) * float(scaling)
        old_weight = reference_weights[plain_name].to(
            device=delta.device,
            dtype=delta.dtype,
        )

        trace_value = torch.sum(old_weight * delta)
        normalized_trace = trace_value / (
            torch.linalg.norm(old_weight).clamp_min(eps)
            * torch.linalg.norm(delta).clamp_min(eps)
        )

        penalties.append(normalized_trace.pow(2))

    if not penalties:
        return torch.tensor(0.0, device=device)

    return torch.stack(penalties).mean()

def compute_orth_diagnostics(model, reference_weights, eps=1e-8):
    """
    Lightweight diagnostics for the current orthogonality objective.
    This does not change the training loss.
    """
    rows = []

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if plain_name not in reference_weights:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        delta = (B @ A) * float(scaling)
        old_weight = reference_weights[plain_name].to(
            device=delta.device,
            dtype=delta.dtype,
        )

        trace_value = torch.sum(old_weight * delta)
        normalized_trace = trace_value / (
            torch.linalg.norm(old_weight).clamp_min(eps)
            * torch.linalg.norm(delta).clamp_min(eps)
        )

        rows.append({
            "layer": plain_name,
            "raw_trace": float(trace_value.detach().cpu()),
            "normalized_trace": float(normalized_trace.detach().cpu()),
            "squared_penalty": float(normalized_trace.pow(2).detach().cpu()),
            "delta_norm": float(torch.linalg.norm(delta).detach().cpu()),
            "reference_norm": float(torch.linalg.norm(old_weight).detach().cpu()),
        })

    if len(rows) == 0:
        print("[orth diagnostics] no matched LoRA/reference layers")
        return pd.DataFrame()

    diag_df = pd.DataFrame(rows)
    summary = diag_df[
        [
            "raw_trace",
            "normalized_trace",
            "squared_penalty",
            "delta_norm",
            "reference_norm",
        ]
    ].mean()

    print("[orth diagnostics] mean over matched q_proj/v_proj layers")
    print(summary.round(6))

    return diag_df

class OrthogonalLossTrainer(Trainer):
    """
    Trainer for the orthogonal_loss variant.

    It adds an orthogonality-style penalty between:
    - previous absorbed model weights M_(t-1)
    - current LoRA delta DeltaW_t
    """

    def __init__(self, *args, reference_weights=None, lambda_orth=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.reference_weights = reference_weights or {}
        self.lambda_orth = float(lambda_orth)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        orth_loss = compute_orth_penalty(
            model=model,
            reference_weights=self.reference_weights,
        )

        loss = ce_loss + self.lambda_orth * orth_loss

        return (loss, outputs) if return_outputs else loss

if METHODS_TO_RUN["orthogonal_loss"]:
    orth_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        print(f"\n===== orthogonal_loss | step {step_idx + 1}/{NUM_STEPS} =====")

        orth_peft_model = add_lora(orth_model)
        orth_peft_model.print_trainable_parameters()

        reference_weights = extract_reference_weights_for_orth(orth_peft_model)

        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        train_with_trainer(
            model=orth_peft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=os.path.join(MODELS_DIR, f"orthogonal_loss_step_{step_idx + 1}"),
            epochs=ORTH_EPOCHS,
            lr=LR_ORTH,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=OrthogonalLossTrainer,
            reference_weights=reference_weights,
            lambda_orth=LAMBDA_ORTH,
        )

        if ORTH_DIAGNOSTICS:
            compute_orth_diagnostics(
                model=orth_peft_model,
                reference_weights=reference_weights,
            )

        orth_model = orth_peft_model.merge_and_unload()

        del orth_peft_model
        cleanup()

    evaluate_model(orth_model, "orthogonal_loss")

    del orth_model
    cleanup()

else:
    print("Skipping orthogonal_loss")


In [ ]:
# ============================================================
# 15) Rank extension: one growing LoRA with copied/frozen old rank blocks
# ============================================================

from transformers import TrainerCallback

class GrowingRankLoRALinear(nn.Module):
    """
    Custom growing-rank LoRA layer.

    This implements growing-rank LoRA as one extended LoRA pair per
    target layer, not as independent accumulated LoRA blocks.

    Step 1: rank = 8.
    Step 2: rank = 16, copy full rank-8 A/B into the old slice and freeze it.
    Step 3: rank = 24, copy full rank-16 A/B into the old slice and freeze it.

    The effective update is always one matrix product:
        DeltaW_extended_t = B_extended_t @ A_extended_t
    """

    def __init__(self, base_layer, total_rank, frozen_A=None, frozen_B=None, dropout=0.0):
        super().__init__()
        self.base_layer = base_layer
        self.total_rank = int(total_rank)

        if frozen_A is None or frozen_B is None:
            self.frozen_rank = 0
        else:
            if frozen_A.shape[0] != frozen_B.shape[1]:
                raise ValueError(
                    f"A/B frozen rank mismatch: A={tuple(frozen_A.shape)}, "
                    f"B={tuple(frozen_B.shape)}"
                )
            self.frozen_rank = int(frozen_A.shape[0])

        self.new_rank = self.total_rank - self.frozen_rank
        if self.new_rank < 0:
            raise ValueError(
                f"new_rank cannot be negative. "
                f"total_rank={self.total_rank}, frozen_rank={self.frozen_rank}"
            )

        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        for p in self.base_layer.parameters():
            p.requires_grad = False

        self.rankext_alpha = RANKEXT_ALPHA_PER_RANK * self.total_rank
        self.scaling = self.rankext_alpha / self.total_rank
        self.dropout = nn.Dropout(dropout)

        if self.frozen_rank > 0:
            if tuple(frozen_A.shape) != (self.frozen_rank, self.in_features):
                raise ValueError(
                    f"Frozen A has wrong shape: {tuple(frozen_A.shape)} "
                    f"expected {(self.frozen_rank, self.in_features)}"
                )
            if tuple(frozen_B.shape) != (self.out_features, self.frozen_rank):
                raise ValueError(
                    f"Frozen B has wrong shape: {tuple(frozen_B.shape)} "
                    f"expected {(self.out_features, self.frozen_rank)}"
                )

            self.A_frozen = nn.Parameter(frozen_A.detach().clone().float(), requires_grad=False)
            self.B_frozen = nn.Parameter(frozen_B.detach().clone().float(), requires_grad=False)
        else:
            self.A_frozen = None
            self.B_frozen = None

        if self.new_rank > 0:
            self.A_new = nn.Parameter(torch.zeros(self.new_rank, self.in_features))
            self.B_new = nn.Parameter(torch.zeros(self.out_features, self.new_rank))
            nn.init.kaiming_uniform_(self.A_new, a=np.sqrt(5))
            nn.init.zeros_(self.B_new)
        else:
            self.A_new = None
            self.B_new = None

    def full_A_B(self):
        """Return one extended A/B pair: copied frozen slice + new slice."""
        A_parts = []
        B_parts = []

        if self.frozen_rank > 0:
            A_parts.append(self.A_frozen.to(device=self.base_layer.weight.device, dtype=self.base_layer.weight.dtype))
            B_parts.append(self.B_frozen.to(device=self.base_layer.weight.device, dtype=self.base_layer.weight.dtype))

        if self.new_rank > 0:
            A_parts.append(self.A_new)
            B_parts.append(self.B_new)

        if len(A_parts) == 0 or len(B_parts) == 0:
            raise ValueError("No LoRA rank blocks available.")

        A = torch.cat(A_parts, dim=0)
        B = torch.cat(B_parts, dim=1)

        if A.shape[0] != self.total_rank or B.shape[1] != self.total_rank:
            raise ValueError(
                f"Extended A/B rank mismatch: A={tuple(A.shape)}, "
                f"B={tuple(B.shape)}, total_rank={self.total_rank}"
            )

        return A, B

    def current_new_delta(self):
        if self.new_rank <= 0:
            return None
        return (self.B_new @ self.A_new) * float(self.scaling)

    def forward(self, x):
        base_out = self.base_layer(x)
        A, B = self.full_A_B()
        x_dropped = self.dropout(x)
        lora_hidden = torch.matmul(x_dropped, A.T)
        lora_out = torch.matmul(lora_hidden, B.T)
        return base_out + self.scaling * lora_out

def get_parent_module_and_child_name(model, module_name):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]

def find_clip_target_linear_modules(model):
    target_names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and (name.endswith("q_proj") or name.endswith("v_proj")):
            target_names.append(name)
    return target_names

def build_rank_extension_model(previous_rank_state=None, step_idx=0):
    """
    Build CLIP-ViT with one growing LoRA per q_proj/v_proj layer.

    The previous full A/B pair is copied into the frozen old-rank slice. Only
    A_new/B_new are trainable for the newly added rank slice.
    """
    model = fresh_pretrained_model()

    for _, p in model.vision_model.named_parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    total_rank = RANKEXT_NEW_RANK_PER_STEP * (step_idx + 1)
    target_names = find_clip_target_linear_modules(model)
    print(f"[rank_extension] Step {step_idx + 1}")
    print(f"  total_rank: {total_rank}")
    print(f"  target linear modules: {len(target_names)}")

    model._rank_extension_target_names = list(target_names)

    for module_name in target_names:
        parent, child_name = get_parent_module_and_child_name(model, module_name)
        base_layer = getattr(parent, child_name)

        frozen_A = None
        frozen_B = None
        if previous_rank_state is not None and module_name in previous_rank_state["lora"]:
            frozen_A = previous_rank_state["lora"][module_name]["A"]
            frozen_B = previous_rank_state["lora"][module_name]["B"]

        setattr(
            parent,
            child_name,
            GrowingRankLoRALinear(
                base_layer=base_layer,
                total_rank=total_rank,
                frozen_A=frozen_A,
                frozen_B=frozen_B,
                dropout=LORA_DROPOUT,
            ),
        )

    if previous_rank_state is not None and previous_rank_state["classifier_weight"] is not None:
        with torch.no_grad():
            model.classifier.weight.copy_(
                previous_rank_state["classifier_weight"].to(
                    device=model.classifier.weight.device,
                    dtype=model.classifier.weight.dtype,
                )
            )
            model.classifier.bias.copy_(
                previous_rank_state["classifier_bias"].to(
                    device=model.classifier.bias.device,
                    dtype=model.classifier.bias.dtype,
                )
            )

    return model

def extract_rank_extension_state(model):
    state = {"lora": {}, "classifier_weight": None, "classifier_bias": None}
    for name, module in model.named_modules():
        if isinstance(module, GrowingRankLoRALinear):
            A, B = module.full_A_B()
            state["lora"][name] = {
                "A": A.detach().cpu().clone(),
                "B": B.detach().cpu().clone(),
                "scaling": float(module.scaling),
                "total_rank": int(module.total_rank),
                "frozen_rank": int(module.frozen_rank),
                "new_rank": int(module.new_rank),
                "rankext_alpha": float(module.rankext_alpha),
            }
    state["classifier_weight"] = model.classifier.weight.detach().cpu().clone()
    state["classifier_bias"] = model.classifier.bias.detach().cpu().clone()
    return state

def rank_extension_trainable_classifier_classes(step_idx, replay_per_class):
    classes = list(classes_for_step(step_idx))
    if replay_per_class > 0:
        for old_step in range(step_idx):
            classes.extend(classes_for_step(old_step))
    return sorted(set(int(c) for c in classes))

def add_classifier_row_gradient_mask(model, trainable_classes):
    """Allow classifier gradients only for rows represented in this training set."""
    trainable_classes = set(int(c) for c in trainable_classes)
    mask_w = torch.zeros_like(model.classifier.weight)
    mask_b = torch.zeros_like(model.classifier.bias)
    for c in trainable_classes:
        mask_w[c, :] = 1.0
        mask_b[c] = 1.0
    hook_w = model.classifier.weight.register_hook(lambda grad: grad * mask_w.to(device=grad.device, dtype=grad.dtype))
    hook_b = model.classifier.bias.register_hook(lambda grad: grad * mask_b.to(device=grad.device, dtype=grad.dtype))
    return [hook_w, hook_b]

def snapshot_protected_classifier_rows(model, trainable_classes):
    trainable_classes = set(int(c) for c in trainable_classes)
    protected_rows = [c for c in range(NUM_CLASSES) if c not in trainable_classes]
    return {
        "rows": protected_rows,
        "weight": model.classifier.weight.detach().cpu().clone(),
        "bias": model.classifier.bias.detach().cpu().clone(),
    }

def restore_protected_classifier_rows(model, snapshot):
    rows = snapshot["rows"]
    if len(rows) == 0:
        return
    with torch.no_grad():
        row_idx = torch.tensor(rows, device=model.classifier.weight.device, dtype=torch.long)
        model.classifier.weight[row_idx].copy_(
            snapshot["weight"][rows].to(device=model.classifier.weight.device, dtype=model.classifier.weight.dtype)
        )
        model.classifier.bias[row_idx].copy_(
            snapshot["bias"][rows].to(device=model.classifier.bias.device, dtype=model.classifier.bias.dtype)
        )

def classifier_protected_row_max_diff(model, snapshot):
    rows = snapshot["rows"]
    if len(rows) == 0:
        return 0.0
    with torch.no_grad():
        weight_diff = (model.classifier.weight.detach().cpu()[rows] - snapshot["weight"][rows]).abs().max().item()
        bias_diff = (model.classifier.bias.detach().cpu()[rows] - snapshot["bias"][rows]).abs().max().item()
    return max(weight_diff, bias_diff)

def snapshot_classifier_rows(model, rows):
    rows = sorted(set(int(r) for r in rows))
    return {
        "rows": rows,
        "weight": model.classifier.weight.detach().cpu().clone(),
        "bias": model.classifier.bias.detach().cpu().clone(),
    }

def classifier_row_diff_diagnostics(model, snapshot, label, csv_path=None):
    rows = snapshot["rows"]
    if len(rows) == 0:
        print(f"[rank_extension diagnostics] {label}: no classifier rows to compare")
        return pd.DataFrame()

    with torch.no_grad():
        weight_now = model.classifier.weight.detach().cpu()[rows]
        bias_now = model.classifier.bias.detach().cpu()[rows]
        weight_before = snapshot["weight"][rows]
        bias_before = snapshot["bias"][rows]

        row_weight_diff = (weight_now - weight_before).abs().amax(dim=1)
        row_bias_diff = (bias_now - bias_before).abs()

    diag_df = pd.DataFrame({
        "class_id": rows,
        "weight_max_abs_diff": row_weight_diff.numpy(),
        "bias_abs_diff": row_bias_diff.numpy(),
    })
    diag_df["row_max_abs_diff"] = diag_df[["weight_max_abs_diff", "bias_abs_diff"]].max(axis=1)

    max_diff = diag_df["row_max_abs_diff"].max()
    print(f"[rank_extension diagnostics] {label}: classifier row max diff={max_diff:.10f}")

    if csv_path is not None:
        diag_df.to_csv(csv_path, index=False)
        print("[rank_extension diagnostics] saved classifier-row diagnostics:", csv_path)

    return diag_df

class ClassifierRowRestoreCallback(TrainerCallback):
    """Restore protected classifier rows after optimizer steps to block AdamW drift."""
    def __init__(self, snapshot):
        self.snapshot = snapshot

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if model is not None:
            restore_protected_classifier_rows(model, self.snapshot)
        return control

    def on_train_end(self, args, state, control, model=None, **kwargs):
        if model is not None:
            restore_protected_classifier_rows(model, self.snapshot)
        return control

class RankExtensionTrainer(Trainer):
    def __init__(self, *args, classifier_snapshot=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.classifier_snapshot = classifier_snapshot
        if classifier_snapshot is not None:
            self.add_callback(ClassifierRowRestoreCallback(classifier_snapshot))

class RankExtensionOrthTrainer(RankExtensionTrainer):
    def __init__(
        self,
        *args,
        reference_weights=None,
        lambda_orth=0.1,
        orth_loss_type="trace",
        orth_scale_mode="squared_trace",
        orth_target_ratio=0.01,
        orth_lambda_min=1e-6,
        orth_lambda_max=1e3,
        orth_eps=1e-8,
        qr_prev_bases=None,
        use_ipc=False,
        lambda_ipc=0.0,
        ipc_protected_layers=None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.reference_weights = reference_weights or {}
        self.lambda_orth = float(lambda_orth)
        self.orth_loss_type = str(orth_loss_type)
        self.orth_scale_mode = str(orth_scale_mode)
        self.orth_target_ratio = float(orth_target_ratio)
        self.orth_lambda_min = float(orth_lambda_min)
        self.orth_lambda_max = float(orth_lambda_max)
        self.orth_eps = float(orth_eps)
        self.qr_prev_bases = qr_prev_bases or {}
        self.use_ipc = bool(use_ipc)
        self.lambda_ipc = float(lambda_ipc)
        self.ipc_protected_layers = set(ipc_protected_layers or [])

        self._ce_losses = []
        self._trace_losses = []
        self._qr_losses = []
        self._orth_losses = []
        self._orth_raw_losses = []
        self._orth_abs_losses = []
        self._orth_squared_losses = []
        self._orth_used_losses = []
        self._orth_contrib_losses = []
        self._orth_ratio_losses = []
        self._effective_lambdas = []
        self._mean_delta_norms = []
        self._mean_previous_weight_norms = []
        self._mean_cosine_alignments = []
        self._raw_trace_unnormalized = []
        self._ipc_losses = []
        self._total_losses = []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        orth_comps = compute_rank_extension_orth_components(
            model=model,
            reference_weights=self.reference_weights,
            eps=self.orth_eps,
        )
        trace_loss = orth_comps["orth_loss_squared"]
        qr_loss = compute_rank_extension_qr_orth_penalty(model=model, qr_prev_bases=self.qr_prev_bases)

        if self.orth_loss_type == "trace":
            orth_loss = trace_loss
        elif self.orth_loss_type == "qr_lorac":
            orth_loss = qr_loss
        elif self.orth_loss_type == "combined":
            orth_loss = 0.5 * (trace_loss + qr_loss)
        else:
            raise ValueError(
                f"Unknown orth_loss_type={self.orth_loss_type}. Use trace, qr_lorac, or combined."
            )

        if self.orth_scale_mode == "raw_trace":
            orth_loss_used = orth_comps["orth_loss_raw"]
        elif self.orth_scale_mode == "abs_trace":
            orth_loss_used = orth_comps["orth_loss_abs"]
        elif self.orth_scale_mode == "squared_trace":
            orth_loss_used = orth_comps["orth_loss_squared"]
        elif self.orth_scale_mode == "cosine":
            orth_loss_used = orth_comps["mean_cosine_alignment"]
        elif self.orth_scale_mode == "auto_ratio":
            orth_loss_used = orth_comps["orth_loss_abs"]
        else:
            raise ValueError(
                f"Unknown orth_scale_mode={self.orth_scale_mode}. "
                f"Use raw_trace, abs_trace, squared_trace, cosine, or auto_ratio."
            )

        ipc_loss = torch.tensor(0.0, device=ce_loss.device, dtype=ce_loss.dtype)
        if self.use_ipc and self.lambda_ipc > 0.0 and len(self.ipc_protected_layers) > 0:
            ipc_loss = compute_rank_extension_ipc_penalty(
                model=model,
                protected_layers=self.ipc_protected_layers,
            )

        effective_lambda = torch.tensor(self.lambda_orth, device=ce_loss.device, dtype=ce_loss.dtype)
        if self.orth_scale_mode == "auto_ratio" and self.orth_target_ratio > 0.0:
            denom = orth_loss_used.detach().abs().clamp_min(self.orth_eps)
            target = float(self.orth_target_ratio) * ce_loss.detach()
            effective_lambda = target / denom
            effective_lambda = effective_lambda.clamp(min=self.orth_lambda_min, max=self.orth_lambda_max)
            effective_lambda = effective_lambda * float(self.lambda_orth)

        orth_contrib = effective_lambda * orth_loss_used
        loss = ce_loss + orth_contrib + self.lambda_ipc * ipc_loss

        self._ce_losses.append(float(ce_loss.detach().cpu().item()))
        self._trace_losses.append(float(trace_loss.detach().cpu().item()))
        self._qr_losses.append(float(qr_loss.detach().cpu().item()))
        self._orth_losses.append(float(orth_loss.detach().cpu().item()))
        self._orth_raw_losses.append(float(orth_comps["orth_loss_raw"].detach().cpu().item()))
        self._orth_abs_losses.append(float(orth_comps["orth_loss_abs"].detach().cpu().item()))
        self._orth_squared_losses.append(float(orth_comps["orth_loss_squared"].detach().cpu().item()))
        self._orth_used_losses.append(float(orth_loss_used.detach().cpu().item()))
        self._orth_contrib_losses.append(float(orth_contrib.detach().cpu().item()))
        orth_ratio = abs(float(orth_contrib.detach().cpu().item())) / max(float(ce_loss.detach().cpu().item()), self.orth_eps)
        self._orth_ratio_losses.append(float(orth_ratio))
        self._effective_lambdas.append(float(effective_lambda.detach().cpu().item()))
        self._mean_delta_norms.append(float(orth_comps["mean_delta_norm"].detach().cpu().item()))
        self._mean_previous_weight_norms.append(float(orth_comps["mean_previous_weight_norm"].detach().cpu().item()))
        self._mean_cosine_alignments.append(float(orth_comps["mean_cosine_alignment"].detach().cpu().item()))
        self._raw_trace_unnormalized.append(float(orth_comps["raw_trace_mean_unnormalized"].detach().cpu().item()))
        self._ipc_losses.append(float(ipc_loss.detach().cpu().item()))
        self._total_losses.append(float(loss.detach().cpu().item()))

        return (loss, outputs) if return_outputs else loss

    def consume_logged_losses(self):
        if len(self._ce_losses) == 0:
            return None

        ce_mean = float(np.mean(self._ce_losses))
        orth_used_mean = float(np.mean(self._orth_used_losses))
        eff_lam_mean = float(np.mean(self._effective_lambdas))
        orth_contrib_mean = float(np.mean(self._orth_contrib_losses))
        ratio_from_means = abs(orth_contrib_mean) / max(ce_mean, self.orth_eps)

        stats = {
            "ce_loss_mean": ce_mean,
            "trace_loss_mean": float(np.mean(self._trace_losses)),
            "qr_loss_mean": float(np.mean(self._qr_losses)),
            "orth_loss_mean": float(np.mean(self._orth_losses)),
            "orth_loss_raw_mean": float(np.mean(self._orth_raw_losses)),
            "orth_loss_abs_mean": float(np.mean(self._orth_abs_losses)),
            "orth_loss_squared_mean": float(np.mean(self._orth_squared_losses)),
            "orth_loss_used_mean": orth_used_mean,
            "orth_contribution_mean": orth_contrib_mean,
            "orth_contribution_ratio_mean": float(np.mean(self._orth_ratio_losses)),
            "orth_contribution_ratio_from_means": float(ratio_from_means),
            "effective_lambda_mean": eff_lam_mean,
            "mean_delta_norm": float(np.mean(self._mean_delta_norms)),
            "mean_previous_weight_norm": float(np.mean(self._mean_previous_weight_norms)),
            "mean_cosine_alignment": float(np.mean(self._mean_cosine_alignments)),
            "raw_trace_unnormalized_mean": float(np.mean(self._raw_trace_unnormalized)),
            "ipc_loss_mean": float(np.mean(self._ipc_losses)),
            "total_loss_mean": float(np.mean(self._total_losses)),
            "num_logged_steps": int(len(self._ce_losses)),
            "lambda_orth": float(self.lambda_orth),
            "orth_loss_type": self.orth_loss_type,
            "orth_scale_mode": self.orth_scale_mode,
            "orth_target_ratio": float(self.orth_target_ratio),
            "orth_lambda_min": float(self.orth_lambda_min),
            "orth_lambda_max": float(self.orth_lambda_max),
            "use_ipc": bool(self.use_ipc),
            "lambda_ipc": float(self.lambda_ipc),
            "ipc_protected_layers": int(len(self.ipc_protected_layers)),
        }

        self._ce_losses = []
        self._trace_losses = []
        self._qr_losses = []
        self._orth_losses = []
        self._orth_raw_losses = []
        self._orth_abs_losses = []
        self._orth_squared_losses = []
        self._orth_used_losses = []
        self._orth_contrib_losses = []
        self._orth_ratio_losses = []
        self._effective_lambdas = []
        self._mean_delta_norms = []
        self._mean_previous_weight_norms = []
        self._mean_cosine_alignments = []
        self._raw_trace_unnormalized = []
        self._ipc_losses = []
        self._total_losses = []

        return stats

def print_rank_extension_trainable_parameters(model, method_name=None, step_idx=None):
    trainable = 0
    total = 0
    rows = []
    lora_rows = []

    for name, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
            row = {"name": name, "numel": p.numel()}
            rows.append(row)
            if ".A_" in name or ".B_" in name:
                lora_rows.append(row)

    pct = 100.0 * trainable / total
    print(f"[rank_extension] trainable params: {trainable:,} || all params: {total:,} || trainable%: {pct:.4f}")

    if RANKEXT_DIAGNOSTICS:
        print("[rank_extension] trainable parameter names/counts")
        for row in rows:
            print(f"  {row['name']} | {row['numel']:,}")

        print("[rank_extension] trainable LoRA parameter names/counts")
        for row in lora_rows:
            print(f"  {row['name']} | {row['numel']:,}")

        if method_name is not None and step_idx is not None:
            trainable_path = os.path.join(
                TABLES_DIR,
                f"{method_name}_step_{step_idx + 1}_trainable_parameters.csv",
            )
            pd.DataFrame(rows).to_csv(trainable_path, index=False)
            print("[rank_extension diagnostics] saved trainable parameters:", trainable_path)

def print_rank_extension_structure_diagnostics(model):
    rows = []
    for name, module in model.named_modules():
        if isinstance(module, GrowingRankLoRALinear):
            rows.append({
                "layer": name,
                "total_rank": module.total_rank,
                "frozen_rank": module.frozen_rank,
                "new_rank": module.new_rank,
                "A_frozen_requires_grad": None if module.A_frozen is None else module.A_frozen.requires_grad,
                "B_frozen_requires_grad": None if module.B_frozen is None else module.B_frozen.requires_grad,
                "A_new_requires_grad": None if module.A_new is None else module.A_new.requires_grad,
                "B_new_requires_grad": None if module.B_new is None else module.B_new.requires_grad,
            })
    diag_df = pd.DataFrame(rows)
    if len(diag_df) == 0:
        print("[rank_extension diagnostics] no GrowingRankLoRALinear layers found")
        return diag_df
    print("[rank_extension diagnostics] rank structure")
    print(
        diag_df[
            [
                "total_rank",
                "frozen_rank",
                "new_rank",
                "A_frozen_requires_grad",
                "B_frozen_requires_grad",
                "A_new_requires_grad",
                "B_new_requires_grad",
            ]
        ].drop_duplicates().to_string(index=False)
    )
    return diag_df

def assert_rank_extension_structure(model, step_idx):
    expected_total_rank = RANKEXT_NEW_RANK_PER_STEP * (step_idx + 1)
    expected_frozen_rank = RANKEXT_NEW_RANK_PER_STEP * step_idx
    expected_new_rank = RANKEXT_NEW_RANK_PER_STEP
    expected_target_names = getattr(model, "_rank_extension_target_names", None)

    if expected_target_names is None:
        raise AssertionError("Missing _rank_extension_target_names on rank-extension model.")

    module_map = dict(model.named_modules())
    wrapped_names = [
        name
        for name, module in module_map.items()
        if isinstance(module, GrowingRankLoRALinear)
    ]

    if sorted(wrapped_names) != sorted(expected_target_names):
        missing = sorted(set(expected_target_names) - set(wrapped_names))
        extra = sorted(set(wrapped_names) - set(expected_target_names))
        raise AssertionError(
            f"Rank-extension wrapper mismatch. missing={missing}, extra={extra}"
        )

    for name in expected_target_names:
        module = module_map[name]
        if not isinstance(module, GrowingRankLoRALinear):
            raise AssertionError(f"{name} is not a GrowingRankLoRALinear.")

        if module.total_rank != expected_total_rank:
            raise AssertionError(
                f"{name} total_rank={module.total_rank}, expected={expected_total_rank}"
            )
        if module.frozen_rank != expected_frozen_rank:
            raise AssertionError(
                f"{name} frozen_rank={module.frozen_rank}, expected={expected_frozen_rank}"
            )
        if module.new_rank != expected_new_rank:
            raise AssertionError(
                f"{name} new_rank={module.new_rank}, expected={expected_new_rank}"
            )

        if module.A_frozen is not None and module.A_frozen.requires_grad:
            raise AssertionError(f"{name}.A_frozen unexpectedly requires grad.")
        if module.B_frozen is not None and module.B_frozen.requires_grad:
            raise AssertionError(f"{name}.B_frozen unexpectedly requires grad.")
        if module.A_new is None or not module.A_new.requires_grad:
            raise AssertionError(f"{name}.A_new is missing or frozen.")
        if module.B_new is None or not module.B_new.requires_grad:
            raise AssertionError(f"{name}.B_new is missing or frozen.")

    trainable_lora_names = [
        name
        for name, p in model.named_parameters()
        if p.requires_grad and (".A_" in name or ".B_" in name)
    ]
    bad_lora_names = [
        name
        for name in trainable_lora_names
        if not (name.endswith(".A_new") or name.endswith(".B_new"))
    ]
    if bad_lora_names:
        raise AssertionError(
            f"Only A_new/B_new may be trainable LoRA parameters, got {bad_lora_names}"
        )

    expected_wrapped_count = len(expected_target_names)
    print(
        f"[rank_extension assertions] step={step_idx + 1} | "
        f"wrapped_layers={len(wrapped_names)}/{expected_wrapped_count} | "
        f"total_rank={expected_total_rank} | "
        f"frozen_rank={expected_frozen_rank} | "
        f"new_rank={expected_new_rank}"
    )

    return True

def snapshot_frozen_rank_blocks(model):
    snapshot = {}
    for name, module in model.named_modules():
        if isinstance(module, GrowingRankLoRALinear) and module.frozen_rank > 0:
            snapshot[name] = {
                "A": module.A_frozen.detach().cpu().clone(),
                "B": module.B_frozen.detach().cpu().clone(),
            }
    return snapshot

def check_frozen_rank_blocks_unchanged(model, snapshot, label, csv_path=None):
    if len(snapshot) == 0:
        print(f"[rank_extension diagnostics] {label}: no frozen rank blocks to compare")
        return pd.DataFrame()

    rows = []
    module_map = dict(model.named_modules())

    for name, before in snapshot.items():
        module = module_map[name]
        rows.append({
            "layer": name,
            "A_max_abs_diff": float((module.A_frozen.detach().cpu() - before["A"]).abs().max().item()),
            "B_max_abs_diff": float((module.B_frozen.detach().cpu() - before["B"]).abs().max().item()),
        })

    diag_df = pd.DataFrame(rows)
    max_a = diag_df["A_max_abs_diff"].max()
    max_b = diag_df["B_max_abs_diff"].max()
    print(f"[rank_extension diagnostics] {label}: max frozen A diff={max_a:.10f}, max frozen B diff={max_b:.10f}")

    if csv_path is not None:
        diag_df.to_csv(csv_path, index=False)
        print("[rank_extension diagnostics] saved frozen-block diagnostics:", csv_path)

    return diag_df

def extract_rank_extension_reference_weights(model):
    reference_weights = {}
    for name, module in model.named_modules():
        if not isinstance(module, GrowingRankLoRALinear):
            continue
        ref = module.base_layer.weight.detach().cpu().float().clone()
        if module.frozen_rank > 0:
            frozen_delta = (module.B_frozen @ module.A_frozen) * float(module.scaling)
            ref = ref + frozen_delta.detach().cpu().float()
        reference_weights[name] = ref
    return reference_weights

def compute_rank_extension_orth_components(model, reference_weights, eps=1e-8):
    rows = []
    raw_trace_terms = []
    cosine_terms = []
    delta_norm_terms = []
    reference_norm_terms = []
    device = next(model.parameters()).device

    for name, module in model.named_modules():
        if not isinstance(module, GrowingRankLoRALinear):
            continue
        if name not in reference_weights or module.new_rank <= 0:
            continue
        delta = module.current_new_delta()
        old_weight = reference_weights[name].to(device=delta.device, dtype=delta.dtype)

        raw_trace = torch.sum(old_weight * delta)
        delta_norm = torch.linalg.norm(delta).clamp_min(eps)
        reference_norm = torch.linalg.norm(old_weight).clamp_min(eps)
        cosine = raw_trace / (reference_norm * delta_norm)

        raw_trace_terms.append(raw_trace)
        cosine_terms.append(cosine)
        delta_norm_terms.append(delta_norm)
        reference_norm_terms.append(reference_norm)

        rows.append({
            "layer": name,
            "raw_trace": float(raw_trace.detach().cpu()),
            "normalized_trace": float(cosine.detach().cpu()),
            "abs_normalized_trace": float(cosine.abs().detach().cpu()),
            "squared_penalty": float(cosine.pow(2).detach().cpu()),
            "delta_norm": float(delta_norm.detach().cpu()),
            "reference_norm": float(reference_norm.detach().cpu()),
        })

    if len(cosine_terms) == 0:
        zero = torch.tensor(0.0, device=device)
        return {
            "num_layers": 0,
            "orth_loss_raw": zero,
            "orth_loss_abs": zero,
            "orth_loss_squared": zero,
            "mean_cosine_alignment": zero,
            "mean_delta_norm": zero,
            "mean_previous_weight_norm": zero,
            "diag_df": pd.DataFrame(),
        }

    raw_tensor = torch.stack(raw_trace_terms)
    cosine_tensor = torch.stack(cosine_terms)
    delta_norm_tensor = torch.stack(delta_norm_terms)
    reference_norm_tensor = torch.stack(reference_norm_terms)

    return {
        "num_layers": int(cosine_tensor.numel()),
        "orth_loss_raw": cosine_tensor.mean(),
        "orth_loss_abs": cosine_tensor.abs().mean(),
        "orth_loss_squared": cosine_tensor.pow(2).mean(),
        "mean_cosine_alignment": cosine_tensor.mean(),
        "mean_delta_norm": delta_norm_tensor.mean(),
        "mean_previous_weight_norm": reference_norm_tensor.mean(),
        "raw_trace_mean_unnormalized": raw_tensor.mean(),
        "diag_df": pd.DataFrame(rows),
    }

def compute_rank_extension_orth_penalty(model, reference_weights, eps=1e-8):
    comps = compute_rank_extension_orth_components(model=model, reference_weights=reference_weights, eps=eps)
    return comps["orth_loss_squared"]

def compute_rank_extension_orth_diagnostics(model, reference_weights, eps=1e-8):
    comps = compute_rank_extension_orth_components(model=model, reference_weights=reference_weights, eps=eps)
    diag_df = comps["diag_df"]
    if diag_df.empty:
        print("[rank_extension orth diagnostics] no matched layers")
        return pd.DataFrame()

    summary = diag_df[[
        "raw_trace",
        "normalized_trace",
        "abs_normalized_trace",
        "squared_penalty",
        "delta_norm",
        "reference_norm",
    ]].mean()
    print("[rank_extension orth diagnostics] mean over growing q_proj/v_proj layers")
    print(summary.round(6))
    return diag_df

def qr_basis_from_A(A, eps=1e-8):
    """
    Build a column-space basis from LoRA A.
    A is expected shape [r, in_features]; QR is run on A.T -> [in_features, r].
    """
    A_t = A.T
    if A_t.numel() == 0:
        return None

    Q, R = torch.linalg.qr(A_t, mode="reduced")
    diag = torch.abs(torch.diagonal(R)) if R.numel() > 0 else torch.tensor([], device=Q.device, dtype=Q.dtype)
    if diag.numel() > 0:
        rank_eff = int((diag > eps).sum().item())
        rank_eff = max(rank_eff, 1)
        Q = Q[:, :rank_eff]
    return Q

def build_prev_q_bases_from_rank_state(previous_rank_state, chunk_rank=8, eps=1e-8):
    prev_q_bases = {}
    if previous_rank_state is None:
        return prev_q_bases

    lora_state = previous_rank_state.get("lora", {})
    for layer_name, layer_state in lora_state.items():
        A_full = layer_state["A"].detach().cpu().float()
        q_list = []
        for start in range(0, A_full.shape[0], int(chunk_rank)):
            A_slice = A_full[start:start + int(chunk_rank), :]
            if A_slice.shape[0] == 0:
                continue
            Q = qr_basis_from_A(A_slice, eps=eps)
            if Q is not None:
                q_list.append(Q.detach().cpu().float())
        prev_q_bases[layer_name] = q_list

    return prev_q_bases

def print_q_basis_diagnostics(prev_q_bases, step_idx, max_layers=6):
    rows = []
    for layer_name, q_list in prev_q_bases.items():
        q_shapes = [tuple(q.shape) for q in q_list]
        rows.append({
            "layer": layer_name,
            "num_prev_q": int(len(q_list)),
            "q_shapes": str(q_shapes[:4]),
        })

    if len(rows) == 0:
        print(f"[rank_extension qr] step {step_idx + 1}: no previous Q bases")
        return pd.DataFrame()

    diag_df = pd.DataFrame(rows).sort_values("layer")
    print(f"[rank_extension qr] step {step_idx + 1}: previous Q-basis diagnostics (first {max_layers} layers)")
    print(diag_df.head(max_layers).to_string(index=False))
    return diag_df

def compute_rank_extension_qr_orth_penalty(model, qr_prev_bases, eps=1e-8):
    # LoRAC-inspired QR orthogonality (lightweight approximation, not exact full reproduction).
    penalties = []
    device = next(model.parameters()).device

    for name, module in model.named_modules():
        if not isinstance(module, GrowingRankLoRALinear):
            continue
        if module.new_rank <= 0:
            continue

        prev_q_list = qr_prev_bases.get(name, [])
        if len(prev_q_list) == 0:
            continue

        Q_current = qr_basis_from_A(module.A_new, eps=eps)
        if Q_current is None:
            continue

        layer_terms = []
        for Q_prev in prev_q_list:
            Q_prev_t = Q_prev.to(device=Q_current.device, dtype=Q_current.dtype)
            overlap = torch.matmul(Q_prev_t.T, Q_current)
            layer_terms.append(torch.linalg.norm(overlap, ord="fro").pow(2))

        if len(layer_terms) > 0:
            penalties.append(torch.stack(layer_terms).mean())

    if len(penalties) == 0:
        return torch.tensor(0.0, device=device)
    return torch.stack(penalties).mean()

def compute_rank_extension_ipc_penalty(model, protected_layers, eps=1e-8):
    penalties = []
    device = next(model.parameters()).device

    for name, module in model.named_modules():
        if not isinstance(module, GrowingRankLoRALinear):
            continue
        if name not in protected_layers or module.new_rank <= 0:
            continue

        delta = module.current_new_delta()
        penalties.append(torch.linalg.norm(delta, ord="fro").pow(2))

    if len(penalties) == 0:
        return torch.tensor(0.0, device=device)
    return torch.stack(penalties).mean()

def estimate_rank_extension_layer_importance(trainer, model, max_batches=8):
    """
    Lightweight IPC-inspired importance estimate using gradient*parameter magnitude.
    """
    layer_scores = {}
    layer_counts = {}
    dataloader = trainer.get_train_dataloader()
    model.train()

    max_batches = max(1, int(max_batches))
    for batch_idx, batch in enumerate(dataloader):
        if batch_idx >= max_batches:
            break

        batch = trainer._prepare_inputs(batch)
        model.zero_grad(set_to_none=True)
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        for name, module in model.named_modules():
            if not isinstance(module, GrowingRankLoRALinear):
                continue
            if module.new_rank <= 0 or module.A_new is None or module.B_new is None:
                continue
            if module.A_new.grad is None or module.B_new.grad is None:
                continue

            a_term = (module.A_new.grad.detach() * module.A_new.detach()).abs().mean().item()
            b_term = (module.B_new.grad.detach() * module.B_new.detach()).abs().mean().item()
            score = float(a_term + b_term)

            layer_scores[name] = layer_scores.get(name, 0.0) + score
            layer_counts[name] = layer_counts.get(name, 0) + 1

    model.zero_grad(set_to_none=True)

    averaged = {}
    for name, score_sum in layer_scores.items():
        count = max(1, layer_counts.get(name, 1))
        averaged[name] = float(score_sum / count)
    return averaged

def select_ipc_protected_layers(importance_map, top_p=0.10):
    if len(importance_map) == 0:
        return []

    rows = sorted(importance_map.items(), key=lambda kv: kv[1], reverse=True)
    k = max(1, int(np.ceil(len(rows) * float(top_p))))
    return [name for name, _ in rows[:k]]

def evaluate_seen_step_accuracies(model, upto_step_idx):
    args = get_training_args(
        output_dir=os.path.join(MODELS_DIR, "tmp_seen_eval"),
        epochs=1,
        lr=LR_LORA,
        batch_size=BATCH_LORA,
        accum_steps=ACCUM_LORA,
        train_dataset_len=None,
        eval_strategy="no",
    )
    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    step_acc = {}
    for task_step in range(upto_step_idx + 1):
        eval_ds = make_eval_dataset(classes_for_step(task_step))
        out = trainer.evaluate(eval_dataset=eval_ds)
        step_acc[int(task_step)] = float(out["eval_accuracy"])
    return step_acc

def compute_average_forgetting(stepwise_task_accuracies):
    if len(stepwise_task_accuracies) == 0:
        return np.nan

    all_steps = sorted(stepwise_task_accuracies.keys())
    final_step = all_steps[-1]
    forgetting_values = []

    for task_step in all_steps:
        if task_step == final_step:
            continue

        vals = []
        for s in all_steps:
            if s < task_step:
                continue
            if task_step in stepwise_task_accuracies[s]:
                vals.append(stepwise_task_accuracies[s][task_step])

        if len(vals) == 0:
            continue

        best_acc = max(vals)
        final_acc = stepwise_task_accuracies[final_step].get(task_step, np.nan)
        if np.isnan(final_acc):
            continue
        forgetting_values.append(float(best_acc - final_acc))

    if len(forgetting_values) == 0:
        return np.nan
    return float(np.mean(forgetting_values))

def format_lambda_for_name(lambda_orth):
    lam = float(lambda_orth)
    if lam == 0.0:
        return "0"
    return f"{lam:g}"

def format_orth_config_tag(
    orth_loss_type,
    lambda_orth,
    use_ipc=False,
    lambda_ipc=0.0,
    orth_scale_mode=None,
    orth_target_ratio=None,
):
    lam_tag = format_lambda_for_name(lambda_orth)
    base = f"{orth_loss_type}_lam_{lam_tag}"
    if orth_scale_mode is not None:
        base = f"{base}_{orth_scale_mode}"
    if orth_target_ratio is not None and float(orth_target_ratio) > 0.0:
        base = f"{base}_tr_{format_lambda_for_name(orth_target_ratio)}"
    if use_ipc:
        ipc_tag = format_lambda_for_name(lambda_ipc)
        base = f"{base}_ipc_{ipc_tag}"
    return base

def run_rank_extension_variant(
    method_name,
    replay_per_class=0,
    use_orth=False,
    orth_config=None,
    orth_eval_records=None,
    orth_train_records=None,
    orth_summary_records=None,
):
    active_orth_loss_type = ORTH_LOSS_TYPE
    active_lambda_orth = float(LAMBDA_ORTH)
    active_orth_scale_mode = str(ORTH_SCALE_MODE)
    active_orth_target_ratio = float(ORTH_TARGET_RATIO)
    active_use_ipc = bool(USE_IPC_CONSTRAINT)
    active_lambda_ipc = float(LAMBDA_IPC)

    if orth_config is not None:
        active_orth_loss_type = str(orth_config.get("orth_loss_type", active_orth_loss_type))
        active_lambda_orth = float(orth_config.get("lambda_orth", active_lambda_orth))
        active_orth_scale_mode = str(orth_config.get("orth_scale_mode", active_orth_scale_mode))
        active_orth_target_ratio = float(orth_config.get("orth_target_ratio", active_orth_target_ratio))
        active_use_ipc = bool(orth_config.get("use_ipc", active_use_ipc))
        active_lambda_ipc = float(orth_config.get("lambda_ipc", active_lambda_ipc))

    previous_rank_state = None
    ipc_protected_layers = set()
    stepwise_task_accuracies = {}

    for step_idx in range(NUM_STEPS):
        current_classes = classes_for_step(step_idx)
        trainable_classifier_classes = rank_extension_trainable_classifier_classes(
            step_idx=step_idx,
            replay_per_class=replay_per_class,
        )
        model = build_rank_extension_model(previous_rank_state=previous_rank_state, step_idx=step_idx)
        assert_rank_extension_structure(model=model, step_idx=step_idx)
        print_rank_extension_trainable_parameters(
            model=model,
            method_name=method_name,
            step_idx=step_idx,
        )
        if RANKEXT_DIAGNOSTICS:
            structure_df = print_rank_extension_structure_diagnostics(model)
            structure_path = os.path.join(
                TABLES_DIR,
                f"{method_name}_step_{step_idx + 1}_rank_structure.csv",
            )
            structure_df.to_csv(structure_path, index=False)
            print("[rank_extension diagnostics] saved rank structure:", structure_path)

        hooks = add_classifier_row_gradient_mask(model=model, trainable_classes=trainable_classifier_classes)
        classifier_snapshot = snapshot_protected_classifier_rows(
            model=model,
            trainable_classes=trainable_classifier_classes,
        )
        frozen_snapshot = snapshot_frozen_rank_blocks(model)
        old_classes = [
            c
            for old_step in range(step_idx)
            for c in classes_for_step(old_step)
        ]
        old_classifier_snapshot = snapshot_classifier_rows(model, old_classes)

        prev_q_bases = build_prev_q_bases_from_rank_state(
            previous_rank_state=previous_rank_state,
            chunk_rank=RANKEXT_NEW_RANK_PER_STEP,
        )
        if use_orth:
            print_q_basis_diagnostics(prev_q_bases=prev_q_bases, step_idx=step_idx)

        if active_use_ipc and len(ipc_protected_layers) > 0:
            protected_preview = sorted(list(ipc_protected_layers))[:8]
            print(
                f"[rank_extension ipc] step {step_idx + 1}: protected layers from previous steps "
                f"= {len(ipc_protected_layers)} | preview={protected_preview}"
            )

        reference_weights = None
        trainer_cls = RankExtensionTrainer
        trainer_kwargs = {"classifier_snapshot": classifier_snapshot}
        if use_orth:
            reference_weights = extract_rank_extension_reference_weights(model)
            trainer_cls = RankExtensionOrthTrainer
            trainer_kwargs.update({
                "reference_weights": reference_weights,
                "lambda_orth": active_lambda_orth,
                "orth_loss_type": active_orth_loss_type,
                "orth_scale_mode": active_orth_scale_mode,
                "orth_target_ratio": active_orth_target_ratio,
                "orth_lambda_min": float(ORTH_LAMBDA_MIN),
                "orth_lambda_max": float(ORTH_LAMBDA_MAX),
                "orth_eps": float(ORTH_EPS),
                "qr_prev_bases": prev_q_bases,
                "use_ipc": active_use_ipc,
                "lambda_ipc": active_lambda_ipc,
                "ipc_protected_layers": ipc_protected_layers,
            })

        train_ds = make_train_dataset(step_idx=step_idx, replay_per_class=replay_per_class)
        eval_ds = make_eval_dataset(current_classes)
        out_dir = os.path.join(MODELS_DIR, f"{method_name}_step_{step_idx + 1}")

        print(
            f"\n===== {method_name} | "
            f"step {step_idx + 1}/{NUM_STEPS} | "
            f"rank={(step_idx + 1) * RANKEXT_NEW_RANK_PER_STEP} | "
            f"frozen_rank={0 if previous_rank_state is None else step_idx * RANKEXT_NEW_RANK_PER_STEP} | "
            f"new_rank={RANKEXT_NEW_RANK_PER_STEP} | "
            f"replay_per_class={replay_per_class} | "
            f"orth={use_orth} | "
            f"orth_loss_type={active_orth_loss_type} | "
            f"orth_scale_mode={active_orth_scale_mode} | "
            f"orth_target_ratio={active_orth_target_ratio:.6g} | "
            f"lambda_orth={active_lambda_orth:.6g} | "
            f"use_ipc={active_use_ipc} | "
            f"lambda_ipc={active_lambda_ipc:.6g} | "
            f"alpha={RANKEXT_ALPHA_PER_RANK * ((step_idx + 1) * RANKEXT_NEW_RANK_PER_STEP):.1f} | "
            f"scaling={RANKEXT_ALPHA_PER_RANK:.2f} ====="
        )
        print(f"[rank_extension] classifier trainable rows: {len(trainable_classifier_classes)} / {NUM_CLASSES}")

        trainer, _ = train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=RANKEXT_EPOCHS,
            lr=LR_RANKEXT,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=trainer_cls,
            **trainer_kwargs,
        )

        if use_orth and isinstance(trainer, RankExtensionOrthTrainer):
            orth_stats = trainer.consume_logged_losses()
            if orth_stats is not None:
                print(
                    f"[rank_extension orth] step {step_idx + 1} | "
                    f"orth_loss_type={orth_stats['orth_loss_type']} | "
                    f"orth_scale_mode={orth_stats['orth_scale_mode']} | "
                    f"orth_target_ratio={orth_stats['orth_target_ratio']:.6g} | "
                    f"lambda_orth={orth_stats['lambda_orth']:.6g} | "
                    f"effective_lambda_mean={orth_stats['effective_lambda_mean']:.6g} | "
                    f"ce_loss_mean={orth_stats['ce_loss_mean']:.6f} | "
                    f"orth_loss_raw_mean={orth_stats['orth_loss_raw_mean']:.6f} | "
                    f"orth_loss_abs_mean={orth_stats['orth_loss_abs_mean']:.6f} | "
                    f"orth_loss_squared_mean={orth_stats['orth_loss_squared_mean']:.6f} | "
                    f"orth_loss_used_mean={orth_stats['orth_loss_used_mean']:.6f} | "
                    f"orth_contribution_mean={orth_stats['orth_contribution_mean']:.6f} | "
                    f"orth_contribution_ratio_mean={orth_stats['orth_contribution_ratio_mean']:.6f} | "
                    f"orth_contribution_ratio_from_means={orth_stats['orth_contribution_ratio_from_means']:.6f} | "
                    f"mean_delta_norm={orth_stats['mean_delta_norm']:.6f} | "
                    f"mean_previous_weight_norm={orth_stats['mean_previous_weight_norm']:.6f} | "
                    f"mean_cosine_alignment={orth_stats['mean_cosine_alignment']:.6f} | "
                    f"trace_loss_mean={orth_stats['trace_loss_mean']:.6f} | "
                    f"qr_loss_mean={orth_stats['qr_loss_mean']:.6f} | "
                    f"orth_loss_mean={orth_stats['orth_loss_mean']:.6f} | "
                    f"ipc_loss_mean={orth_stats['ipc_loss_mean']:.6f} | "
                    f"total_loss_mean={orth_stats['total_loss_mean']:.6f} | "
                    f"logged_steps={orth_stats['num_logged_steps']}"
                )

                if orth_train_records is not None:
                    orth_train_records.append({
                        "method": method_name,
                        "step": int(step_idx + 1),
                        "orth_loss_type": orth_stats["orth_loss_type"],
                        "orth_scale_mode": orth_stats["orth_scale_mode"],
                        "orth_target_ratio": float(orth_stats["orth_target_ratio"]),
                        "lambda_orth": float(orth_stats["lambda_orth"]),
                        "effective_lambda_mean": float(orth_stats["effective_lambda_mean"]),
                        "ce_loss_mean": float(orth_stats["ce_loss_mean"]),
                        "orth_loss_raw_mean": float(orth_stats["orth_loss_raw_mean"]),
                        "orth_loss_abs_mean": float(orth_stats["orth_loss_abs_mean"]),
                        "orth_loss_squared_mean": float(orth_stats["orth_loss_squared_mean"]),
                        "orth_loss_used_mean": float(orth_stats["orth_loss_used_mean"]),
                        "orth_contribution_mean": float(orth_stats["orth_contribution_mean"]),
                        "orth_contribution_ratio_mean": float(orth_stats["orth_contribution_ratio_mean"]),
                        "orth_contribution_ratio_from_means": float(orth_stats["orth_contribution_ratio_from_means"]),
                        "mean_delta_norm": float(orth_stats["mean_delta_norm"]),
                        "mean_previous_weight_norm": float(orth_stats["mean_previous_weight_norm"]),
                        "mean_cosine_alignment": float(orth_stats["mean_cosine_alignment"]),
                        "raw_trace_unnormalized_mean": float(orth_stats["raw_trace_unnormalized_mean"]),
                        "trace_loss_mean": float(orth_stats["trace_loss_mean"]),
                        "qr_loss_mean": float(orth_stats["qr_loss_mean"]),
                        "orth_loss_mean": float(orth_stats["orth_loss_mean"]),
                        "use_ipc": bool(orth_stats["use_ipc"]),
                        "lambda_ipc": float(orth_stats["lambda_ipc"]),
                        "ipc_loss_mean": float(orth_stats["ipc_loss_mean"]),
                        "ipc_protected_layers": int(orth_stats["ipc_protected_layers"]),
                        "total_loss_mean": float(orth_stats["total_loss_mean"]),
                        "logged_steps": int(orth_stats["num_logged_steps"]),
                    })

        if use_orth and active_use_ipc:
            importance_map = estimate_rank_extension_layer_importance(
                trainer=trainer,
                model=model,
                max_batches=IPC_IMPORTANCE_NUM_BATCHES,
            )
            protected_now = select_ipc_protected_layers(importance_map=importance_map, top_p=IPC_TOP_P)
            print(
                f"[rank_extension ipc] step {step_idx + 1}: selected protected layers {len(protected_now)} / {len(importance_map)}"
            )
            print(f"[rank_extension ipc] step {step_idx + 1}: {protected_now}")
            ipc_protected_layers.update(protected_now)

        classifier_row_diff_diagnostics(
            model=model,
            snapshot=old_classifier_snapshot,
            label=f"{method_name} step {step_idx + 1} old rows before final restore",
            csv_path=os.path.join(
                TABLES_DIR,
                f"{method_name}_step_{step_idx + 1}_old_classifier_rows_before_restore.csv",
            ),
        )
        restore_protected_classifier_rows(model, classifier_snapshot)
        protected_diff = classifier_protected_row_max_diff(model, classifier_snapshot)
        print(f"[rank_extension diagnostics] protected classifier row max diff after restore: {protected_diff:.10f}")
        classifier_row_diff_diagnostics(
            model=model,
            snapshot=old_classifier_snapshot,
            label=f"{method_name} step {step_idx + 1} old rows after restore",
            csv_path=os.path.join(
                TABLES_DIR,
                f"{method_name}_step_{step_idx + 1}_old_classifier_rows_after_restore.csv",
            ),
        )
        check_frozen_rank_blocks_unchanged(
            model=model,
            snapshot=frozen_snapshot,
            label=f"{method_name} step {step_idx + 1}",
            csv_path=os.path.join(
                TABLES_DIR,
                f"{method_name}_step_{step_idx + 1}_frozen_rank_blocks.csv",
            ),
        )

        if use_orth and ORTH_DIAGNOSTICS:
            compute_rank_extension_orth_diagnostics(model=model, reference_weights=reference_weights)

        if use_orth and orth_summary_records is not None:
            seen_acc = evaluate_seen_step_accuracies(model=model, upto_step_idx=step_idx)
            stepwise_task_accuracies[int(step_idx)] = seen_acc

        for h in hooks:
            h.remove()

        previous_rank_state = extract_rank_extension_state(model)
        del model
        cleanup()

    final_rank_model = build_rank_extension_model(previous_rank_state=previous_rank_state, step_idx=NUM_STEPS - 1)
    eval_rows = evaluate_model(final_rank_model, method_name)

    avg_forgetting = compute_average_forgetting(stepwise_task_accuracies)
    if use_orth:
        print(
            f"[rank_extension orth] method={method_name} | avg_forgetting={avg_forgetting if not np.isnan(avg_forgetting) else 'nan'}"
        )

    if use_orth and orth_eval_records is not None:
        for row in eval_rows:
            orth_eval_records.append({
                "method": row["method"],
                "eval_set": row["eval_set"],
                "accuracy": row["accuracy"],
                "loss": row["loss"],
                "orth_loss_type": active_orth_loss_type,
                "orth_scale_mode": active_orth_scale_mode,
                "orth_target_ratio": float(active_orth_target_ratio),
                "lambda_orth": float(active_lambda_orth),
                "use_ipc": bool(active_use_ipc),
                "lambda_ipc": float(active_lambda_ipc),
                "ipc_top_p": float(IPC_TOP_P),
                "avg_forgetting": float(avg_forgetting) if not np.isnan(avg_forgetting) else np.nan,
            })

    if use_orth and orth_summary_records is not None:
        eval_map = {row["eval_set"]: float(row["accuracy"]) for row in eval_rows}
        orth_summary_records.append({
            "method": method_name,
            "orth_loss_type": active_orth_loss_type,
            "orth_scale_mode": active_orth_scale_mode,
            "orth_target_ratio": float(active_orth_target_ratio),
            "lambda_orth": float(active_lambda_orth),
            "use_ipc": bool(active_use_ipc),
            "lambda_ipc": float(active_lambda_ipc),
            "ipc_top_p": float(IPC_TOP_P),
            "first_step": eval_map.get("first_step", np.nan),
            "later_steps": eval_map.get("later_steps", np.nan),
            "all_seen": eval_map.get("all_seen", np.nan),
            "old_new_gap": eval_map.get("first_step", np.nan) - eval_map.get("later_steps", np.nan),
            "avg_forgetting": float(avg_forgetting) if not np.isnan(avg_forgetting) else np.nan,
        })
    del final_rank_model
    cleanup()

rank_extension_variants = [
    ("rank_extension", 0, False),
    ("rank_extension_replay", RANKEXT_REPLAY_PER_CLASS, False),
    ("rank_extension_orth", 0, True),
    ("rank_extension_replay_orth", RANKEXT_REPLAY_PER_CLASS, True),
]

orth_sweep_eval_rows = []
orth_sweep_train_rows = []
orth_sweep_summary_rows = []

for method_name, replay_per_class, use_orth in rank_extension_variants:
    if not METHODS_TO_RUN.get(method_name, False):
        print(f"Skipping {method_name}")
        continue

    if method_name == "rank_extension_orth" and use_orth and ENABLE_RANKEXT_ORTH_CONFIG_SWEEP:
        print("\n===== rank_extension_orth config sweep =====")
        print("Sweep configs:", ORTH_CONFIG_SWEEP)
        for cfg in ORTH_CONFIG_SWEEP:
            cfg_tag = format_orth_config_tag(
                orth_loss_type=cfg.get("orth_loss_type", ORTH_LOSS_TYPE),
                lambda_orth=cfg.get("lambda_orth", LAMBDA_ORTH),
                use_ipc=cfg.get("use_ipc", False),
                lambda_ipc=cfg.get("lambda_ipc", 0.0),
                orth_scale_mode=cfg.get("orth_scale_mode", ORTH_SCALE_MODE),
                orth_target_ratio=cfg.get("orth_target_ratio", ORTH_TARGET_RATIO),
            )
            sweep_method_name = f"rank_extension_orth_{cfg_tag}"
            run_rank_extension_variant(
                method_name=sweep_method_name,
                replay_per_class=replay_per_class,
                use_orth=True,
                orth_config=cfg,
                orth_eval_records=orth_sweep_eval_rows,
                orth_train_records=orth_sweep_train_rows,
                orth_summary_records=orth_sweep_summary_rows,
            )
    else:
        run_rank_extension_variant(
            method_name=method_name,
            replay_per_class=replay_per_class,
            use_orth=use_orth,
            orth_eval_records=orth_sweep_eval_rows if (use_orth and ENABLE_RANKEXT_ORTH_CONFIG_SWEEP) else None,
            orth_train_records=orth_sweep_train_rows if (use_orth and ENABLE_RANKEXT_ORTH_CONFIG_SWEEP) else None,
            orth_summary_records=orth_sweep_summary_rows if (use_orth and ENABLE_RANKEXT_ORTH_CONFIG_SWEEP) else None,
        )

if len(orth_sweep_eval_rows) > 0:
    orth_sweep_results_df = pd.DataFrame(orth_sweep_eval_rows)
    orth_sweep_results_path = os.path.join(TABLES_DIR, "orth_lambda_sweep_results.csv")
    orth_sweep_results_df.to_csv(orth_sweep_results_path, index=False)
    print("[rank_extension orth] saved lambda sweep eval results:", orth_sweep_results_path)

    if len(orth_sweep_summary_rows) > 0:
        orth_sweep_summary = pd.DataFrame(orth_sweep_summary_rows)
    else:
        orth_sweep_summary = orth_sweep_results_df.pivot_table(
            index="lambda_orth",
            columns="eval_set",
            values="accuracy",
            aggfunc="mean",
        ).reset_index()
        for col in ["first_step", "later_steps", "all_seen"]:
            if col not in orth_sweep_summary.columns:
                orth_sweep_summary[col] = np.nan
        orth_sweep_summary["old_new_gap"] = orth_sweep_summary["first_step"] - orth_sweep_summary["later_steps"]
        orth_sweep_summary["avg_forgetting"] = np.nan
        orth_sweep_summary["orth_loss_type"] = "unknown"
        orth_sweep_summary["orth_scale_mode"] = "unknown"
        orth_sweep_summary["orth_target_ratio"] = np.nan
        orth_sweep_summary["use_ipc"] = False
        orth_sweep_summary["lambda_ipc"] = 0.0
        orth_sweep_summary["ipc_top_p"] = IPC_TOP_P
        orth_sweep_summary["method"] = "unknown"

    orth_sweep_summary = orth_sweep_summary.sort_values(
        ["orth_loss_type", "orth_scale_mode", "orth_target_ratio", "use_ipc", "lambda_orth", "lambda_ipc"]
    ).reset_index(drop=True)

    orth_sweep_summary_percent = orth_sweep_summary.copy()
    for col in ["first_step", "later_steps", "all_seen", "old_new_gap", "avg_forgetting"]:
        if col in orth_sweep_summary_percent.columns:
            orth_sweep_summary_percent[col] = orth_sweep_summary_percent[col] * 100.0

    orth_sweep_summary_path = os.path.join(TABLES_DIR, "orth_lambda_sweep_summary.csv")
    orth_sweep_summary_percent.to_csv(orth_sweep_summary_path, index=False)
    print("[rank_extension orth] saved lambda sweep summary:", orth_sweep_summary_path)
    display(orth_sweep_summary_percent.round(3))

    line_df = orth_sweep_summary_percent.copy()
    line_df["config_label"] = line_df.apply(
        lambda row: format_orth_config_tag(
            orth_loss_type=row.get("orth_loss_type", "unknown"),
            lambda_orth=row.get("lambda_orth", 0.0),
            use_ipc=bool(row.get("use_ipc", False)),
            lambda_ipc=row.get("lambda_ipc", 0.0),
        ),
        axis=1,
    )

    metric_plot_specs = [
        ("all_seen", "08_orth_lambda_all_seen.png", "All-seen Accuracy across orth configs", "All-seen accuracy (%)"),
        ("first_step", "09_orth_lambda_first_step.png", "First-step Accuracy across orth configs", "First-step accuracy (%)"),
        ("later_steps", "10_orth_lambda_later_steps.png", "Later-steps Accuracy across orth configs", "Later-steps accuracy (%)"),
        ("old_new_gap", "11_orth_lambda_old_new_gap.png", "Old-New Gap across orth configs", "first_step - later_steps (%)"),
    ]

    for metric_col, fname, title, ylabel in metric_plot_specs:
        y = line_df[metric_col].values
        plt.figure(figsize=(8, 5))
        plt.plot(line_df["config_label"], y, marker="o")
        plt.xlabel("orth config")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.xticks(rotation=35, ha="right")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        out_path = os.path.join(PLOTS_DIR, fname)
        plt.savefig(out_path, dpi=200)
        plt.show()
        print("Saved:", out_path)

if len(orth_sweep_train_rows) > 0:
    orth_train_df = pd.DataFrame(orth_sweep_train_rows)
    orth_train_path = os.path.join(TABLES_DIR, "orth_lambda_sweep_train_losses.csv")
    orth_train_df.to_csv(orth_train_path, index=False)
    print("[rank_extension orth] saved lambda sweep training losses:", orth_train_path)


In [ ]:
# ============================================================
# 14) joint_upper_bound
# ============================================================

if METHODS_TO_RUN["joint_upper_bound"]:
    joint_model = fresh_pretrained_model()

    train_joint = make_joint_train_dataset()
    test_joint = make_joint_eval_dataset()

    print("\n===== joint_upper_bound =====")

    train_with_trainer(
        model=joint_model,
        train_ds=train_joint,
        eval_ds=test_joint,
        output_dir=os.path.join(MODELS_DIR, "joint_upper_bound"),
        epochs=JOINT_EPOCHS,
        lr=LR_JOINT,
        batch_size=BATCH_FT,
        accum_steps=ACCUM_FT,
    )

    evaluate_model(joint_model, "joint_upper_bound")

    del joint_model
    cleanup()

else:
    print("Skipping joint_upper_bound")

In [ ]:
# ============================================================
# 15) Final results table
# ============================================================

results_df = pd.DataFrame(all_results)

results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_debug.csv")
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
results_df

In [ ]:
# ============================================================
# 18) Final pivot table + summary metrics
# ============================================================

method_order = [
    "seq_ft_no_replay",
    "simple_avg_no_replay",
    "simple_avg_replay",
    "do_merging_simple",
    "orthogonal_loss",
    "rank_extension",
    "rank_extension_replay",
    "rank_extension_orth",
    "rank_extension_replay_orth",
    "joint_upper_bound",
]

results_df = pd.DataFrame(all_results)
results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_full_comparison.csv")
results_df.to_csv(results_path, index=False)

print("Saved raw results:", results_path)
display(results_df)

final_table = results_df.pivot_table(index="method", columns="eval_set", values="accuracy", aggfunc="mean")

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in final_table.columns:
        final_table[col] = np.nan

ordered_methods = [m for m in method_order if m in final_table.index]
extra_methods = sorted([m for m in final_table.index if m not in method_order])
final_table = final_table.reindex(ordered_methods + extra_methods)
final_table = final_table[["first_step", "later_steps", "all_seen"]]
final_table_percent = final_table * 100
summary_table = final_table_percent.copy()

summary_table["old_new_gap"] = summary_table["first_step"] - summary_table["later_steps"]
summary_table["min_old_new"] = summary_table[["first_step", "later_steps"]].min(axis=1)
summary_table["avg_old_new"] = summary_table[["first_step", "later_steps"]].mean(axis=1)

if "joint_upper_bound" in summary_table.index:
    joint_all_seen = summary_table.loc["joint_upper_bound", "all_seen"]
    summary_table["gap_to_joint_all_seen"] = joint_all_seen - summary_table["all_seen"]
else:
    summary_table["gap_to_joint_all_seen"] = np.nan

if "seq_ft_no_replay" in summary_table.index:
    first_step_reference = summary_table.loc["seq_ft_no_replay", "first_step"]
else:
    first_step_reference = np.nan
summary_table["first_step_drop_vs_seq_ft"] = first_step_reference - summary_table["first_step"]

comparison_rows = []

def add_gain(name, better, base):
    if better in summary_table.index and base in summary_table.index:
        gain = (
            summary_table.loc[better, ["first_step", "later_steps", "all_seen"]]
            - summary_table.loc[base, ["first_step", "later_steps", "all_seen"]]
        )
        comparison_rows.append({
            "comparison": name,
            "first_step_gain": gain["first_step"],
            "later_steps_gain": gain["later_steps"],
            "all_seen_gain": gain["all_seen"],
        })
        print(f"\n{name}")
        print(gain.round(2))

add_gain("Replay gain: simple_avg_replay - simple_avg_no_replay", "simple_avg_replay", "simple_avg_no_replay")
add_gain("Replay gain: rank_extension_replay - rank_extension", "rank_extension_replay", "rank_extension")
add_gain("Orthogonality gain: rank_extension_orth - rank_extension", "rank_extension_orth", "rank_extension")
add_gain(
    "Orthogonality gain with replay: rank_extension_replay_orth - rank_extension_replay",
    "rank_extension_replay_orth",
    "rank_extension_replay",
)
add_gain("Rank-extension gain: rank_extension - simple_avg_no_replay", "rank_extension", "simple_avg_no_replay")
add_gain("Rank-extension replay gain: rank_extension_replay - simple_avg_replay", "rank_extension_replay", "simple_avg_replay")

comparison_table = pd.DataFrame(comparison_rows)

final_table_path = os.path.join(TABLES_DIR, "final_accuracy_clip_vit_percent.csv")
summary_table_path = os.path.join(TABLES_DIR, "summary_metrics_clip_vit_percent.csv")
comparison_table_path = os.path.join(TABLES_DIR, "comparison_gains_clip_vit_percent.csv")

final_table_percent.to_csv(final_table_path)
summary_table.to_csv(summary_table_path)
comparison_table.to_csv(comparison_table_path, index=False)

print("\nSaved final accuracy table:", final_table_path)
print("Saved summary metrics table:", summary_table_path)
print("Saved comparison gains table:", comparison_table_path)

print("\nFinal accuracy (%):")
display(final_table_percent.round(2))

print("\nSummary metrics (%):")
display(summary_table.round(2))

print("\nComparison gains (%):")
display(comparison_table.round(2))


In [ ]:
# ============================================================
# 19) More plots for comparison
# ============================================================

plot_df = final_table_percent.reset_index()

x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, plot_df["first_step"], width, label="first_step")
plt.bar(x, plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, plot_df["method"], rotation=30, ha="right")
plt.ylabel("Accuracy (%)")
plt.title("CLIP-ViT + Split CIFAR-100: Final Accuracy Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "01_grouped_accuracy_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)

plt.figure(figsize=(9, 5))

heatmap_data = final_table_percent[
    ["first_step", "later_steps", "all_seen"]
].copy()

plt.imshow(heatmap_data.values, aspect="auto")
plt.colorbar(label="Accuracy (%)")

plt.xticks(
    ticks=np.arange(len(heatmap_data.columns)),
    labels=heatmap_data.columns,
    rotation=0,
)

plt.yticks(
    ticks=np.arange(len(heatmap_data.index)),
    labels=heatmap_data.index,
)

for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[i, j]
        if not np.isnan(value):
            plt.text(j, i, f"{value:.1f}", ha="center", va="center")

plt.title("Accuracy Heatmap (%)")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "02_accuracy_heatmap.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)

ranking_df = final_table_percent.sort_values("all_seen", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(ranking_df.index, ranking_df["all_seen"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("All-seen accuracy (%)")
plt.title("Method Ranking by All-seen Accuracy")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "03_all_seen_ranking.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)

gap_df = summary_table.copy()

plt.figure(figsize=(10, 5))
plt.bar(gap_df.index, gap_df["old_new_gap"])
plt.axhline(0, linestyle="--", linewidth=1)
plt.xticks(rotation=30, ha="right")
plt.ylabel("first_step - later_steps accuracy (%)")
plt.title("Old-New Accuracy Gap")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "04_old_new_gap.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)

if "gap_to_joint_all_seen" in summary_table.columns:
    gap_joint_df = summary_table.drop(index="joint_upper_bound", errors="ignore")

    plt.figure(figsize=(10, 5))
    plt.bar(gap_joint_df.index, gap_joint_df["gap_to_joint_all_seen"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Gap to joint all-seen accuracy (%)")
    plt.title("Distance from Joint Upper Bound")
    plt.tight_layout()

    plot_path = os.path.join(PLOTS_DIR, "05_gap_to_joint_upper_bound.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print("Saved:", plot_path)

if (
    "simple_avg_no_replay" in final_table_percent.index
    and "simple_avg_replay" in final_table_percent.index
):
    replay_gain_series = (
        final_table_percent.loc["simple_avg_replay", ["first_step", "later_steps", "all_seen"]]
        - final_table_percent.loc["simple_avg_no_replay", ["first_step", "later_steps", "all_seen"]]
    )

    plt.figure(figsize=(7, 5))
    plt.bar(replay_gain_series.index, replay_gain_series.values)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.ylabel("Accuracy gain (%)")
    plt.title("Replay Gain: simple_avg_replay - simple_avg_no_replay")
    plt.tight_layout()

    plot_path = os.path.join(PLOTS_DIR, "06_replay_gain.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print("Saved:", plot_path)

loss_table = results_df.pivot_table(
    index="method",
    columns="eval_set",
    values="loss",
    aggfunc="mean",
)

loss_table = loss_table.reindex(
    [m for m in method_order if m in loss_table.index]
)

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in loss_table.columns:
        loss_table[col] = np.nan

loss_table = loss_table[
    ["first_step", "later_steps", "all_seen"]
]

loss_table_path = os.path.join(TABLES_DIR, "final_loss_table.csv")
loss_table.to_csv(loss_table_path)

loss_plot_df = loss_table.reset_index()

x = np.arange(len(loss_plot_df))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, loss_plot_df["first_step"], width, label="first_step")
plt.bar(x, loss_plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, loss_plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, loss_plot_df["method"], rotation=30, ha="right")
plt.ylabel("Loss")
plt.title("Evaluation Loss Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "07_loss_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved loss table:", loss_table_path)
print("Saved:", plot_path)